
# MNAR Gaussian Model Implementation

This notebook implements the first modelling stage from `Final Year Project Proposal.docx`.

The implementation follows the proposal's structure:

- observed matrix `X_obs` with missing entries represented by `np.nan` unless a mask is supplied;
- known deterministic censoring rule object, either self-censoring intervals or linear threshold rules;
- conditional missing-block distribution as a truncated conditional Gaussian;
- approximation by slow Monte Carlo Gibbs sampling (`approximation_style='SLOW'`), fast deterministic/importance approximations (`approximation_style='FAST'`), or automatic FAST-with-SLOW-fallback routing (`approximation_style='AUTO'`, the default);
- update by projected stochastic gradient descent (`update_style='SGD'`) or expectation maximisation (`update_style='EM'`);
- optional per-iteration L2 and squared-L2 diagnostic history when `true_mean` and `true_covariance` are supplied.

Important API note: the proposal treats the censoring map `h` as known and fixed. Therefore `censoring_rule` is required by the fitting function. If `known_covariance=True`, the covariance is fixed to the supplied `true_covariance`.


In [1]:

from __future__ import annotations

from dataclasses import asdict, dataclass, is_dataclass, replace
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple
import json
import io
import os
import itertools
import time

_mpl_config_dir = Path.cwd() / ".matplotlib-cache"
_mpl_config_dir.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_mpl_config_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import ipywidgets as widgets
except Exception:
    widgets = None
from numpy.typing import ArrayLike
from scipy.optimize import LinearConstraint, linprog, minimize
from scipy.special import expit, logsumexp
from scipy.stats import norm, qmc, truncnorm
from IPython.display import clear_output, display, HTML, Markdown
try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm.auto import tqdm


In [2]:
display(HTML("""
<style>
/* Dark-mode patch for tqdm.notebook / ipywidgets progress bars */
.widget-inline-hbox,
.widget-hbox,
.jp-OutputArea-output .widget-inline-hbox,
.jp-OutputArea-output .widget-hbox {
    background: #000000 !important;
    color: #f5f5f5 !important;
}

.widget-label,
.widget-readout,
.widget-html-content,
.jp-OutputArea-output .widget-label,
.jp-OutputArea-output .widget-readout,
.jp-OutputArea-output .widget-html-content {
    color: #f5f5f5 !important;
}

.widget-progress,
.progress {
    background-color: #1b1b1b !important;
    border: 1px solid #333333 !important;
    box-shadow: none !important;
}

.progress-bar {
    background-color: #00c853 !important;
    color: #000000 !important;
}

.p-Widget,
.lm-Widget,
.jupyter-widgets {
    color: #f5f5f5 !important;
}
</style>
"""))


## Configuration Objects

These dataclasses mirror the optimizer, sampler, and projection parameter tables in the proposal.


In [3]:

@dataclass(frozen=True)
class OptimizerConfig:
    n_iter: int = 100
    batch_size: int = 8
    eta_nu0: float = 1e-2
    eta_T0: float = 1e-3
    decay_lr: bool = True
    snr_epsilon: float = 1e-12
    live_plot: bool = False
    live_plot_every: int = 5
    validation_fraction: float = 0.1
    validation_every: int = 5
    validation_impatience_threshold: Optional[int] = 10
    track_objective_impatience: bool = True
    track_JD_impatience: bool = True
    compute_validation_observed_jd: bool = False
    validation_mask_eta: float = 0.5
    validation_kernel_epsilon: float = 1e-12
    validation_kernel_eig_floor: float = 1e-8


@dataclass(frozen=True)
class SamplerConfig:
    K: int = 5
    burn_in: int = 20
    thin: int = 1
    warm_start: bool = True
    fast_grid_points: int = 20
    fast_max_grid_size: int = 10**4
    fast_tilt_samples: int = 200
    fast_min_feasible: int = 100


@dataclass(frozen=True)
class ProjectionConfig:
    tau_lower: float = 1e-3
    tau_upper: float = 1e3
    use_A2: bool = True
    r_T: Optional[float] = None
    use_A3: bool = True
    r_nu: Optional[float] = None
    covariance_eig_floor: float = 1e-8


@dataclass
class FitResult:
    mean: np.ndarray
    covariance: np.ndarray
    history: Dict[str, list]
    nu: np.ndarray
    precision: np.ndarray
    update_style: str
    approximation_style: str



## Linear Algebra and Diagnostic Utilities


In [4]:

def symmetrize(A: ArrayLike) -> np.ndarray:
    A = np.asarray(A, dtype=float)
    return 0.5 * (A + A.T)


def vech(A: ArrayLike) -> np.ndarray:
    A = np.asarray(A, dtype=float)
    return A[np.tril_indices(A.shape[0])]


def parameter_vector(mu: ArrayLike, Sigma: ArrayLike) -> np.ndarray:
    mu = np.asarray(mu, dtype=float)
    Sigma = np.asarray(Sigma, dtype=float)
    return np.concatenate([mu, vech(Sigma)])


def eigen_clip_spd(
    A: ArrayLike,
    lower: float,
    upper: Optional[float] = None,
) -> np.ndarray:
    A = symmetrize(A)
    vals, vecs = np.linalg.eigh(A)
    vals = np.maximum(vals, lower)
    if upper is not None:
        vals = np.minimum(vals, upper)
    return symmetrize((vecs * vals) @ vecs.T)


def raw_to_canonical(mu: ArrayLike, Sigma: ArrayLike) -> Tuple[np.ndarray, np.ndarray]:
    mu = np.asarray(mu, dtype=float)
    Sigma = np.asarray(Sigma, dtype=float)
    T = np.linalg.inv(Sigma)
    nu = T @ mu
    return nu, T


def canonical_to_raw(nu: ArrayLike, T: ArrayLike) -> Tuple[np.ndarray, np.ndarray]:
    nu = np.asarray(nu, dtype=float)
    T = np.asarray(T, dtype=float)
    Sigma = np.linalg.inv(T)
    mu = Sigma @ nu
    return mu, symmetrize(Sigma)


def infer_mask(X_obs: ArrayLike, mask: Optional[ArrayLike] = None) -> np.ndarray:
    if mask is None:
        return ~np.isnan(np.asarray(X_obs, dtype=float))
    return np.asarray(mask, dtype=bool)


def split_indices(mask_row: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    obs_idx = np.flatnonzero(mask_row)
    miss_idx = np.flatnonzero(~mask_row)
    return obs_idx, miss_idx


def reconstruct_full(
    y: np.ndarray,
    z: np.ndarray,
    obs_idx: np.ndarray,
    miss_idx: np.ndarray,
    d: int,
) -> np.ndarray:
    x = np.empty(d, dtype=float)
    x[obs_idx] = y
    x[miss_idx] = z
    return x


def completed_moments_from_missing(
    y: np.ndarray,
    obs_idx: np.ndarray,
    miss_idx: np.ndarray,
    z_mean: np.ndarray,
    Q_z: np.ndarray,
    d: int,
) -> Tuple[np.ndarray, np.ndarray]:
    x_bar = np.zeros(d, dtype=float)
    Xi = np.zeros((d, d), dtype=float)

    x_bar[obs_idx] = y
    x_bar[miss_idx] = z_mean

    Xi[np.ix_(obs_idx, obs_idx)] = np.outer(y, y)
    Xi[np.ix_(obs_idx, miss_idx)] = np.outer(y, z_mean)
    Xi[np.ix_(miss_idx, obs_idx)] = np.outer(z_mean, y)
    Xi[np.ix_(miss_idx, miss_idx)] = Q_z
    return x_bar, Xi


def gaussian_observed_negative_log_density(
    *,
    y: np.ndarray,
    mu_obs: np.ndarray,
    Sigma_obs: np.ndarray,
    eig_floor: float = 1e-12,
) -> float:
    m = len(y)
    if m == 0:
        return 0.0

    Sigma_obs = symmetrize(Sigma_obs)
    sign, logdet = np.linalg.slogdet(Sigma_obs)
    if sign <= 0 or not np.isfinite(logdet):
        Sigma_obs = eigen_clip_spd(Sigma_obs, lower=eig_floor)
        sign, logdet = np.linalg.slogdet(Sigma_obs)
    if sign <= 0 or not np.isfinite(logdet):
        raise ValueError("Observed covariance block must be positive definite.")

    delta = y - mu_obs
    quadratic = float(delta @ np.linalg.solve(Sigma_obs, delta))
    return float(0.5 * (m * np.log(2.0 * np.pi) + logdet + quadratic))


def parameter_error(
    *,
    update_style: str,
    known_covariance: bool,
    true_mean: np.ndarray,
    true_covariance: np.ndarray,
    nu_est: Optional[np.ndarray] = None,
    T_est: Optional[np.ndarray] = None,
    mu_est: Optional[np.ndarray] = None,
    Sigma_est: Optional[np.ndarray] = None,
) -> Tuple[float, float]:
    if update_style == "SGD":
        mu_est, Sigma_est = canonical_to_raw(nu_est, T_est)
    elif mu_est is None or Sigma_est is None:
        raise ValueError("EM parameter error requires mu_est and Sigma_est.")

    raw_delta = np.concatenate([mu_est - true_mean, vech(Sigma_est - true_covariance)])
    squared_l2 = float(raw_delta @ raw_delta)
    mean_squared_l2 = squared_l2 / raw_delta.size
    return float(np.sqrt(squared_l2)), float(mean_squared_l2)



## Censoring Rules

The proposal has two censoring families:

- self censoring: coordinate `j` is missing exactly when `X_j` lies inside a finite interval `[a_j, b_j]`;
- linear censoring: coordinate `j` is visible exactly when `a_j.T @ X <= b_j`.

The rule object is responsible for feasible-domain membership, one-dimensional Gibbs bounds, and initial feasible projection.


In [5]:

class CensoringRule:
    kind: str

    def visible_mask(self, x: ArrayLike) -> np.ndarray:
        raise NotImplementedError

    def feasible(
        self,
        z: np.ndarray,
        obs_idx: np.ndarray,
        miss_idx: np.ndarray,
        y: np.ndarray,
        mask_row: np.ndarray,
        tol: float = 1e-10,
    ) -> bool:
        raise NotImplementedError

    def project(
        self,
        point: np.ndarray,
        obs_idx: np.ndarray,
        miss_idx: np.ndarray,
        y: np.ndarray,
        mask_row: np.ndarray,
    ) -> np.ndarray:
        raise NotImplementedError

    def coordinate_bounds(
        self,
        local_j: int,
        z: np.ndarray,
        obs_idx: np.ndarray,
        miss_idx: np.ndarray,
        y: np.ndarray,
        mask_row: np.ndarray,
    ) -> Tuple[float, float]:
        raise NotImplementedError

    def initial_point(
        self,
        obs_idx: np.ndarray,
        miss_idx: np.ndarray,
        y: np.ndarray,
        mask_row: np.ndarray,
    ) -> np.ndarray:
        return self.project(np.zeros(len(miss_idx), dtype=float), obs_idx, miss_idx, y, mask_row)
    
    def to_dataframes(self) -> Dict[str, pd.DataFrame]:
        raise NotImplementedError

    def show(self) -> None:
        display(Markdown(f"**Censoring rule kind:** `{self.kind}`"))
        for name, frame in self.to_dataframes().items():
            display(Markdown(f"**{name}**"))
            display(frame)

    def _ipython_display_(self) -> None:
        self.show()


def linear_constraints_feasible(point: np.ndarray, H: np.ndarray, c: np.ndarray, tol: float = 1e-10) -> bool:
    point = np.asarray(point, dtype=float)
    H = np.asarray(H, dtype=float)
    c = np.asarray(c, dtype=float)
    if H.shape[1] == 0:
        return bool(np.all(0.0 <= c + tol))
    return bool(np.all(H @ point <= c + tol))


def find_linear_feasible_point(
    H: np.ndarray,
    c: np.ndarray,
    *,
    tol: float = 1e-7,
    error_prefix: str = "linear constraint feasibility",
) -> np.ndarray:
    H = np.asarray(H, dtype=float)
    c = np.asarray(c, dtype=float)
    if H.ndim != 2:
        raise ValueError(f"{error_prefix}: H must be a 2D matrix.")
    if c.shape != (H.shape[0],):
        raise ValueError(f"{error_prefix}: c must have shape (H.shape[0],).")
    r = H.shape[1]
    if r == 0:
        if np.all(0.0 <= c + tol):
            return np.zeros(0, dtype=float)
        raise ValueError(f"{error_prefix}: zero-dimensional constraint set is infeasible.")

    feasible_start_result = linprog(
        c=np.zeros(r, dtype=float),
        A_ub=H,
        b_ub=c,
        bounds=[(None, None)] * r,
        method="highs",
    )
    if (
        not feasible_start_result.success
        or not np.all(np.isfinite(feasible_start_result.x))
        or not linear_constraints_feasible(feasible_start_result.x, H, c, tol=tol)
    ):
        message = getattr(feasible_start_result, "message", "unknown solver failure")
        raise ValueError(f"{error_prefix}: HiGHS could not find a feasible point ({message}).")
    return np.asarray(feasible_start_result.x, dtype=float)


def project_onto_linear_constraints(
    point: np.ndarray,
    H: np.ndarray,
    c: np.ndarray,
    *,
    tol: float = 1e-7,
    ftol: float = 1e-10,
    maxiter: int = 500,
    error_prefix: str = "linear constraint projection",
) -> np.ndarray:
    point = np.asarray(point, dtype=float)
    H = np.asarray(H, dtype=float)
    c = np.asarray(c, dtype=float)
    if point.ndim != 1:
        raise ValueError(f"{error_prefix}: point must be a 1D vector.")
    if H.ndim != 2:
        raise ValueError(f"{error_prefix}: H must be a 2D matrix.")
    if H.shape[1] != point.size:
        raise ValueError(f"{error_prefix}: H and point dimensions do not match.")
    if c.shape != (H.shape[0],):
        raise ValueError(f"{error_prefix}: c must have shape (H.shape[0],).")
    if point.size == 0:
        if np.all(0.0 <= c + tol):
            return point.copy()
        raise ValueError(f"{error_prefix}: zero-dimensional constraint set is infeasible.")
    if linear_constraints_feasible(point, H, c, tol=tol):
        return point.copy()

    feasible_start = find_linear_feasible_point(H, c, tol=tol, error_prefix=error_prefix)
    constraints = [
        {
            "type": "ineq",
            "fun": lambda v, row=row, rhs=rhs: rhs - row @ v,
            "jac": lambda v, row=row, rhs=rhs: -row,
        }
        for row, rhs in zip(H, c)
    ]
    objective = lambda v: 0.5 * float(np.sum((v - point) ** 2))
    gradient = lambda v: v - point

    slsqp_result = minimize(
        fun=objective,
        x0=feasible_start,
        jac=gradient,
        constraints=constraints,
        method="SLSQP",
        options={"ftol": ftol, "maxiter": maxiter, "disp": False},
    )
    if (
        slsqp_result.success
        and np.all(np.isfinite(slsqp_result.x))
        and linear_constraints_feasible(slsqp_result.x, H, c, tol=tol)
    ):
        return np.asarray(slsqp_result.x, dtype=float)

    linear_constraint = LinearConstraint(H, np.full(H.shape[0], -np.inf), c)
    trust_result = minimize(
        fun=objective,
        x0=feasible_start,
        jac=gradient,
        hess=lambda v: np.eye(point.size),
        constraints=[linear_constraint],
        method="trust-constr",
        options={"gtol": ftol, "xtol": ftol, "barrier_tol": ftol, "maxiter": maxiter, "verbose": 0},
    )
    if (
        trust_result.success
        and np.all(np.isfinite(trust_result.x))
        and linear_constraints_feasible(trust_result.x, H, c, tol=tol)
    ):
        return np.asarray(trust_result.x, dtype=float)

    slsqp_message = getattr(slsqp_result, "message", "unknown SLSQP failure")
    trust_message = getattr(trust_result, "message", "unknown trust-constr failure")
    raise ValueError(
        f"{error_prefix}: projection failed after SLSQP ({slsqp_message}) "
        f"and trust-constr ({trust_message})."
    )


@dataclass(frozen=True)
class SelfCensoringRule(CensoringRule):
    intervals: np.ndarray
    kind: str = "self"

    def __post_init__(self):
        intervals = np.asarray(self.intervals, dtype=float)
        if intervals.ndim != 2 or intervals.shape[1] != 2:
            raise ValueError("intervals must have shape (d, 2).")
        if np.any(intervals[:, 0] > intervals[:, 1]):
            raise ValueError("Each censoring interval must satisfy lower <= upper.")
        object.__setattr__(self, "intervals", intervals)

    def visible_mask(self, x: ArrayLike) -> np.ndarray:
        x = np.asarray(x, dtype=float)
        lo = self.intervals[:, 0]
        hi = self.intervals[:, 1]
        return (x < lo) | (x > hi)

    def feasible(self, z, obs_idx, miss_idx, y, mask_row, tol=1e-10) -> bool:
        if len(miss_idx) == 0:
            return True
        lo = self.intervals[miss_idx, 0]
        hi = self.intervals[miss_idx, 1]
        return bool(np.all(z >= lo - tol) and np.all(z <= hi + tol))

    def project(self, point, obs_idx, miss_idx, y, mask_row) -> np.ndarray:
        point = np.asarray(point, dtype=float)
        lo = self.intervals[miss_idx, 0]
        hi = self.intervals[miss_idx, 1]
        return np.clip(point, lo, hi)

    def initial_point(self, obs_idx, miss_idx, y, mask_row) -> np.ndarray:
        lo = self.intervals[miss_idx, 0]
        hi = self.intervals[miss_idx, 1]
        return 0.5 * (lo + hi)

    def coordinate_bounds(self, local_j, z, obs_idx, miss_idx, y, mask_row) -> Tuple[float, float]:
        original_j = miss_idx[local_j]
        lo, hi = self.intervals[original_j]
        return float(lo), float(hi)
    
    def to_dataframes(self) -> Dict[str, pd.DataFrame]:
        intervals_df = pd.DataFrame(
            self.intervals,
            columns=["lower", "upper"],
            index=[f"x_{j}" for j in range(self.intervals.shape[0])],
        )
        return {"intervals": intervals_df}


@dataclass(frozen=True)
class LinearCensoringRule(CensoringRule):
    A: np.ndarray
    b: np.ndarray
    kind: str = "linear"

    def __post_init__(self):
        A = np.asarray(self.A, dtype=float)
        b = np.asarray(self.b, dtype=float)
        if A.ndim != 2 or A.shape[0] != A.shape[1]:
            raise ValueError("A must be a square matrix with one row a_j.T per coordinate.")
        if b.shape != (A.shape[0],):
            raise ValueError("b must have shape (d,).")
        if np.any(np.linalg.norm(A, axis=1) == 0):
            raise ValueError("Each linear censoring row a_j must be nonzero.")
        object.__setattr__(self, "A", A)
        object.__setattr__(self, "b", b)

    def visible_mask(self, x: ArrayLike) -> np.ndarray:
        x = np.asarray(x, dtype=float)
        return (self.A @ x) <= self.b

    def constraints(
        self,
        obs_idx: np.ndarray,
        miss_idx: np.ndarray,
        y: np.ndarray,
        mask_row: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray]:
        D_diag = 2.0 * mask_row.astype(float) - 1.0
        A_tilde = D_diag[:, None] * self.A
        b_tilde = D_diag * self.b
        H = A_tilde[:, miss_idx]
        c = b_tilde - A_tilde[:, obs_idx] @ y
        return H, c

    def feasible(self, z, obs_idx, miss_idx, y, mask_row, tol=1e-10) -> bool:
        H, c = self.constraints(obs_idx, miss_idx, y, mask_row)
        if H.shape[1] == 0:
            return bool(np.all(0.0 <= c + tol))
        return bool(np.all(H @ z <= c + tol))

    def project(self, point, obs_idx, miss_idx, y, mask_row) -> np.ndarray:
        point = np.asarray(point, dtype=float)
        if len(miss_idx) == 0:
            return point.copy()
        H, c = self.constraints(obs_idx, miss_idx, y, mask_row)
        return project_onto_linear_constraints(
            point,
            H,
            c,
            error_prefix="linear censoring projection",
        )

    def initial_point(self, obs_idx, miss_idx, y, mask_row) -> np.ndarray:
        return self.project(np.zeros(len(miss_idx), dtype=float), obs_idx, miss_idx, y, mask_row)

    def coordinate_bounds(self, local_j, z, obs_idx, miss_idx, y, mask_row) -> Tuple[float, float]:
        H, c = self.constraints(obs_idx, miss_idx, y, mask_row)
        h = H[:, local_j]
        other = H @ z - h * z[local_j]
        rhs = c - other

        lower = -np.inf
        upper = np.inf
        pos = h > 1e-14
        neg = h < -1e-14
        if np.any(pos):
            upper = float(np.min(rhs[pos] / h[pos]))
        if np.any(neg):
            lower = float(np.max(rhs[neg] / h[neg]))
        return lower, upper
    
    def to_dataframes(self) -> Dict[str, pd.DataFrame]:
        d = self.A.shape[0]
        A_df = pd.DataFrame(
            self.A,
            columns=[f"x_{j}" for j in range(d)],
            index=[f"a_{j}" for j in range(d)],
        )
        b_df = pd.DataFrame(
            self.b,
            columns=["b"],
            index=[f"a_{j}" for j in range(d)],
        )
        return {"A": A_df, "b": b_df}



## Censoring Rule Helpers

These helpers are useful for later experiments and ablation runs.


In [6]:

def apply_censoring_rule(
    X_full: ArrayLike,
    censoring_rule: CensoringRule,
    missing_value: float = np.nan,
) -> Tuple[np.ndarray, np.ndarray]:
    X_full = np.asarray(X_full, dtype=float)
    masks = np.vstack([censoring_rule.visible_mask(row) for row in X_full])
    X_obs = X_full.copy()
    X_obs[~masks] = missing_value
    return X_obs, masks


def generate_self_censoring_rule(
    true_mean: ArrayLike,
    true_covariance: ArrayLike,
    alpha_min_pair: float,
    rng: Optional[np.random.Generator] = None,
) -> SelfCensoringRule:
    rng = np.random.default_rng(rng)
    mu = np.asarray(true_mean, dtype=float)
    Sigma = np.asarray(true_covariance, dtype=float)
    if not np.isfinite(alpha_min_pair) or not (0.0 <= alpha_min_pair <= 1.0):
        raise ValueError("alpha_min_pair must be in [0, 1].")
    p0 = 0.5 * (1.0 + alpha_min_pair)

    intervals = np.empty((len(mu), 2), dtype=float)
    for j in range(len(mu)):
        sd = np.sqrt(Sigma[j, j])
        if not np.isfinite(sd) or sd < 0.0:
            raise ValueError("true_covariance must have finite non-negative diagonal entries.")
        if alpha_min_pair == 1.0 or sd == 0.0:
            intervals[j] = mu[j]
            continue
        u_left = rng.uniform(0.0, p0)
        u_right = u_left + 1.0 - p0
        u_left = np.clip(u_left, np.finfo(float).tiny, 1.0 - np.finfo(float).eps)
        u_right = np.clip(u_right, np.finfo(float).tiny, 1.0 - np.finfo(float).eps)
        intervals[j, 0] = mu[j] + sd * norm.ppf(u_left)
        intervals[j, 1] = mu[j] + sd * norm.ppf(u_right)
    return SelfCensoringRule(intervals)


def generate_linear_censoring_rule(
    true_mean: ArrayLike,
    true_covariance: ArrayLike,
    alpha_min_pair: float,
    rng: Optional[np.random.Generator] = None,
) -> LinearCensoringRule:
    rng = np.random.default_rng(rng)
    mu = np.asarray(true_mean, dtype=float)
    Sigma = np.asarray(true_covariance, dtype=float)
    d = len(mu)
    p0 = 0.5 * (1.0 + alpha_min_pair)
    tau = norm.ppf(p0)

    A = rng.normal(size=(d, d))
    row_norms = np.linalg.norm(A, axis=1)
    while np.any(row_norms == 0):
        A[row_norms == 0] = rng.normal(size=(np.sum(row_norms == 0), d))
        row_norms = np.linalg.norm(A, axis=1)

    b = np.empty(d, dtype=float)
    for j in range(d):
        sigma_a = np.sqrt(A[j] @ Sigma @ A[j])
        b[j] = A[j] @ mu + tau * sigma_a
    return LinearCensoringRule(A, b)



## Conditional Gaussian Block

Given canonical parameters `(nu, T)`, the missing block conditional before truncation has canonical parameters

`T_cond = T[miss, miss]`, `nu_cond = nu[miss] - T[miss, obs] @ y`.


In [7]:

def conditional_gaussian_from_canonical(
    nu: np.ndarray,
    T: np.ndarray,
    obs_idx: np.ndarray,
    miss_idx: np.ndarray,
    y: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    r = len(miss_idx)
    if r == 0:
        empty_vec = np.empty(0, dtype=float)
        empty_mat = np.empty((0, 0), dtype=float)
        return empty_vec, empty_mat, empty_vec, empty_mat

    T_cond = T[np.ix_(miss_idx, miss_idx)]
    T_mo = T[np.ix_(miss_idx, obs_idx)]
    nu_cond = nu[miss_idx] - T_mo @ y
    mu_cond = np.linalg.solve(T_cond, nu_cond)
    Sigma_cond = symmetrize(np.linalg.inv(T_cond))
    return mu_cond, Sigma_cond, nu_cond, T_cond



## Monte Carlo Gibbs Sampler

Each Gibbs sweep updates every missing coordinate once from its exact one-dimensional truncated normal conditional. The implementation uses the equivalent precision-form conditional mean, which is algebraically identical to the covariance formula in the proposal for a Gaussian conditional block.


In [8]:

def _dashboard_text(value: Any) -> str:
    text = "" if value is None else str(value)
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


class DashboardProgressBar:
    def __init__(self, label: str, *, width: str = "360px") -> None:
        self.description = label
        self.total = 1
        self.n = 0
        self.last_print_n = 0
        self.is_dashboard_bar = True
        self.label_widget = widgets.HTML(value=f"<b>{_dashboard_text(label)}</b>", layout=widgets.Layout(width="185px"))
        self.progress_widget = widgets.IntProgress(
            value=0,
            min=0,
            max=1,
            bar_style="",
            layout=widgets.Layout(width=width),
        )
        self.status_widget = widgets.HTML(value="", layout=widgets.Layout(width="520px"))
        self.container = widgets.HBox(
            [self.label_widget, self.progress_widget, self.status_widget],
            layout=widgets.Layout(display="none", visibility="hidden", align_items="center"),
        )

    def reset(self, total: int = 1) -> None:
        total = max(1, int(total))
        self.total = total
        self.n = 0
        self.last_print_n = 0
        self.progress_widget.max = total
        self.progress_widget.value = 0
        self.progress_widget.bar_style = ""
        self.status_widget.value = ""
        self.container.layout.display = ""
        self.container.layout.visibility = "visible"

    def set_description_str(self, desc: str, refresh: bool = False) -> None:
        self.description = str(desc)
        self.label_widget.value = f"<b>{_dashboard_text(desc)}</b>"
        if refresh:
            self.refresh()

    def set_postfix(self, ordered_dict: Optional[Dict[str, Any]] = None, refresh: bool = True, **kwargs: Any) -> None:
        items: Dict[str, Any] = {}
        if ordered_dict:
            items.update(ordered_dict)
        items.update(kwargs)
        self.status_widget.value = " | ".join(
            f"{_dashboard_text(key)}={_dashboard_text(value)}" for key, value in items.items()
        )
        if refresh:
            self.refresh()

    def update(self, n: int = 1) -> None:
        self.n = min(self.total, self.n + int(n))
        self.progress_widget.value = int(self.n)
        self.last_print_n = self.n

    def refresh(self) -> None:
        self.progress_widget.value = int(min(self.n, self.total))

    def close(self) -> None:
        self.progress_widget.bar_style = "success"
        if not self.status_widget.value:
            self.status_widget.value = "complete"
        self.container.layout.display = ""
        self.container.layout.visibility = "visible"

    def hide(self) -> None:
        self.progress_widget.bar_style = ""
        self.status_widget.value = ""
        self.container.layout.display = "none"
        self.container.layout.visibility = "hidden"


class DashboardLivePlotHandle:
    def __init__(self, image_widget: widgets.Image, status_widget: widgets.HTML) -> None:
        self.image_widget = image_widget
        self.status_widget = status_widget

    def update(self, obj: Any) -> None:
        self.image_widget.layout.display = ""
        if obj is None:
            self.image_widget.value = b""
            self.status_widget.value = ""
            return
        if hasattr(obj, "savefig"):
            buffer = io.BytesIO()
            obj.savefig(buffer, format="png", dpi=120, bbox_inches="tight", facecolor=obj.get_facecolor())
            self.image_widget.value = buffer.getvalue()
            self.status_widget.value = f"Live plot updated at {time.strftime('%H:%M:%S')}"
        else:
            self.status_widget.value = _dashboard_text(obj)


class ProgressDashboard:
    DEFAULT_LEVELS = (
        ("studies", "Ablation studies"),
        ("classes", "Problem classes"),
        ("problems", "Problems"),
        ("values", "Independent values"),
        ("iterations", "Optimizer iterations"),
    )

    def __init__(self, *, title: str = "Live ablation dashboard") -> None:
        if widgets is None:
            raise ImportError("ipywidgets is required for the persistent progress dashboard.")
        self.title = widgets.HTML(value=f"<h4 style='margin: 0 0 6px 0'>{_dashboard_text(title)}</h4>")
        self.bars = {name: DashboardProgressBar(label) for name, label in self.DEFAULT_LEVELS}
        self.status = widgets.HTML(value="", layout=widgets.Layout(width="100%"))
        self.live_status = widgets.HTML(value="", layout=widgets.Layout(width="100%"))
        self.live_image = widgets.Image(format="png", layout=widgets.Layout(display="none", max_width="100%"))
        self.container = widgets.VBox(
            [self.title, self.status, *(bar.container for bar in self.bars.values()), self.live_status, self.live_image],
            layout=widgets.Layout(width="100%"),
        )
        self.display_handle = None

    def display(self) -> None:
        if self.display_handle is None:
            self.display_handle = display(self.container, display_id=True)
        else:
            self.display_handle.update(self.container)

    def bar(self, name: str) -> DashboardProgressBar:
        if name not in self.bars:
            raise KeyError(f"Unknown dashboard progress bar {name!r}.")
        return self.bars[name]

    def heartbeat(self, message: str = "") -> None:
        self.status.value = f"{_dashboard_text(message)} | {time.strftime('%H:%M:%S')}" if message else time.strftime('%H:%M:%S')

    def live_plot_handle(self) -> DashboardLivePlotHandle:
        self.live_image.layout.display = ""
        return DashboardLivePlotHandle(self.live_image, self.live_status)

    def clear_live_plot(self) -> None:
        self.live_image.value = b""
        self.live_image.layout.display = "none"
        self.live_status.value = ""

    def close_all(self, *, clear_plot: bool = True) -> None:
        for bar in self.bars.values():
            bar.hide()
        if clear_plot:
            self.clear_live_plot()


def reset_progress_bar(progress_bar, *, total: int, desc: str) -> None:
    progress_bar.reset(total=total)
    progress_bar.set_description_str(desc, refresh=False)
    progress_bar.refresh()


def complete_progress_bar(progress_bar) -> None:
    if progress_bar is None:
        return
    total = getattr(progress_bar, "total", None)
    current = getattr(progress_bar, "n", 0)
    if total is not None and current < total:
        progress_bar.n = total
        if hasattr(progress_bar, "last_print_n"):
            progress_bar.last_print_n = total
        progress_bar.refresh()


def close_progress_bar(progress_bar) -> None:
    if progress_bar is None:
        return
    complete_progress_bar(progress_bar)
    container = getattr(progress_bar, "container", None)
    is_dashboard_bar = bool(getattr(progress_bar, "is_dashboard_bar", False))
    if is_dashboard_bar:
        try:
            progress_bar.close()
        except Exception:
            pass
        return
    if container is not None:
        try:
            container.children[1].bar_style = ""
        except Exception:
            pass
    try:
        progress_bar.close()
    except Exception:
        pass
    try:
        progress_bar.display(close=True, check_delay=False)
    except Exception:
        pass
    if container is not None:
        try:
            container.close()
        except Exception:
            pass
        try:
            container.layout.display = "none"
            container.layout.visibility = "hidden"
        except Exception:
            pass


def sample_truncated_normal_icdf(
    mean: float,
    sd: float,
    lower: float,
    upper: float,
    rng: np.random.Generator,
    interval_tol: float = 1e-5,
) -> float:
    if not np.isfinite(sd) or sd <= 0:
        raise ValueError("Truncated normal scale must be positive and finite.")
    finite_bounds = np.isfinite(lower) and np.isfinite(upper)
    interval_scale = max(1.0, abs(lower) if np.isfinite(lower) else 0.0, abs(upper) if np.isfinite(upper) else 0.0)
    collapse_width = interval_tol * interval_scale
    interval_width = upper - lower
    collapsed_interval = finite_bounds and abs(interval_width) <= collapse_width
    if interval_width < 0.0:
        if collapsed_interval:
            return float(0.5 * (lower + upper))
        raise ValueError(f"Empty truncation interval [{lower}, {upper}].")
    if collapsed_interval:
        return float(0.5 * (lower + upper))

    a = (lower - mean) / sd if np.isfinite(lower) else -np.inf
    b = (upper - mean) / sd if np.isfinite(upper) else np.inf
    if b <= a:
        standardized_scale = max(1.0, abs(a) if np.isfinite(a) else 0.0, abs(b) if np.isfinite(b) else 0.0)
        if finite_bounds and abs(a - b) <= interval_tol * standardized_scale:
            return float(0.5 * (lower + upper))
        raise ValueError("Numerically degenerate truncated normal interval.")
    return float(truncnorm.rvs(a, b, loc=mean, scale=sd, random_state=rng))


def gibbs_sample_missing_block(
    *,
    y: np.ndarray,
    mask_row: np.ndarray,
    obs_idx: np.ndarray,
    miss_idx: np.ndarray,
    nu: np.ndarray,
    T: np.ndarray,
    censoring_rule: CensoringRule,
    sampler: SamplerConfig,
    rng: np.random.Generator,
    warm_state: Optional[np.ndarray] = None,
    show_progress: bool = True,
    progress_bar: Optional[object] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    r = len(miss_idx)
    if r == 0:
        return np.empty((sampler.K, 0), dtype=float), np.empty(0, dtype=float)

    mu_cond, _, nu_cond, T_cond = conditional_gaussian_from_canonical(nu, T, obs_idx, miss_idx, y)
    used_warm_state = False
    if warm_state is not None and len(warm_state) == r and censoring_rule.feasible(warm_state, obs_idx, miss_idx, y, mask_row):
        z = np.asarray(warm_state, dtype=float).copy()
        used_warm_state = True
    else:
        try:
            z = censoring_rule.project(mu_cond, obs_idx, miss_idx, y, mask_row)
        except ValueError:
            z = censoring_rule.initial_point(obs_idx, miss_idx, y, mask_row)

    burn_in = 0 if used_warm_state else sampler.burn_in
    retain_sweeps = {burn_in + 1 + k * sampler.thin for k in range(sampler.K)}
    final_sweep = max(retain_sweeps) if retain_sweeps else burn_in
    retained = []

    close_sweep_iter = False
    if progress_bar is not None:
        reset_progress_bar(progress_bar, total=final_sweep, desc="Gibbs sweeps")
        sweep_iter = range(1, final_sweep + 1)
    elif show_progress:
        sweep_iter = tqdm(
            range(1, final_sweep + 1),
            desc="Gibbs sweeps",
            unit="sweep",
            position=2,
            leave=False,
        )
        close_sweep_iter = True
    else:
        sweep_iter = range(1, final_sweep + 1)

    try:
        for sweep in sweep_iter:
            for local_j in range(r):
                precision_jj = T_cond[local_j, local_j]
                offdiag_dot = T_cond[local_j] @ z - precision_jj * z[local_j]
                cond_mean = (nu_cond[local_j] - offdiag_dot) / precision_jj
                cond_sd = np.sqrt(1.0 / precision_jj)
                lower, upper = censoring_rule.coordinate_bounds(local_j, z, obs_idx, miss_idx, y, mask_row)
                try:
                    z[local_j] = sample_truncated_normal_icdf(cond_mean, cond_sd, lower, upper, rng)
                except ValueError as exc:
                    if "truncation interval" not in str(exc) and "truncated normal interval" not in str(exc):
                        raise
                    z = censoring_rule.project(z, obs_idx, miss_idx, y, mask_row)
                    lower, upper = censoring_rule.coordinate_bounds(local_j, z, obs_idx, miss_idx, y, mask_row)
                    z[local_j] = sample_truncated_normal_icdf(cond_mean, cond_sd, lower, upper, rng)

            if sweep in retain_sweeps:
                retained.append(z.copy())
            if progress_bar is not None:
                progress_bar.update(1)
    finally:
        if close_sweep_iter:
            close_progress_bar(sweep_iter)

    return np.asarray(retained, dtype=float), z.copy()


def conditional_completed_moments_slow(
    *,
    X_obs_row: np.ndarray,
    mask_row: np.ndarray,
    nu: np.ndarray,
    T: np.ndarray,
    censoring_rule: CensoringRule,
    sampler: SamplerConfig,
    rng: np.random.Generator,
    warm_state: Optional[np.ndarray] = None,
    show_progress: bool = True,
    progress_bar: Optional[object] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    d = X_obs_row.shape[0]
    obs_idx, miss_idx = split_indices(mask_row)
    y = X_obs_row[obs_idx]

    if len(miss_idx) == 0:
        x = X_obs_row.astype(float).copy()
        return x, np.outer(x, x), np.empty(0, dtype=float)

    z_samples, final_state = gibbs_sample_missing_block(
        y=y,
        mask_row=mask_row,
        obs_idx=obs_idx,
        miss_idx=miss_idx,
        nu=nu,
        T=T,
        censoring_rule=censoring_rule,
        sampler=sampler,
        rng=rng,
        warm_state=warm_state,
        show_progress=show_progress,
        progress_bar=progress_bar,
    )
    z_mean = np.mean(z_samples, axis=0)
    Q_z = np.einsum("ki,kj->ij", z_samples, z_samples) / z_samples.shape[0]
    x_bar, Xi = completed_moments_from_missing(y, obs_idx, miss_idx, z_mean, Q_z, d)
    return x_bar, Xi, final_state



## Fast Approximation Moment Computer

For self-censoring, FAST first uses a midpoint Riemann sum over the missing-coordinate hyperrectangle with `sampler.fast_grid_points` grid points per coordinate, raising if the total grid size exceeds `sampler.fast_max_grid_size`. If Riemann fails, the self-censoring box is rerouted as linear constraints and FAST uses simplified exponential tilting. For linear censoring, FAST uses simplified exponential tilting directly in the conditional standardized Gaussian coordinates, using deterministic cached Sobol-derived standard Gaussian proposals up to `sampler.fast_tilt_samples` and stopping once at least `sampler.fast_min_feasible` feasible proposals have been collected.


In [9]:

_SOBOL_STANDARD_NORMAL_CACHE: Dict[Tuple[int, int], np.ndarray] = {}


def sobol_standard_normals(d: int, n: int) -> np.ndarray:
    if not isinstance(d, (int, np.integer)) or d <= 0:
        raise ValueError("Sobol proposal dimension must be a positive integer.")
    if not isinstance(n, (int, np.integer)) or n <= 0:
        raise ValueError("Sobol proposal count must be a positive integer.")
    key = (int(d), int(n))
    cached = _SOBOL_STANDARD_NORMAL_CACHE.get(key)
    if cached is not None:
        return cached

    n_power = 1 << int(np.ceil(np.log2(n + 1)))
    sobol_engine = qmc.Sobol(d=int(d), scramble=False)
    uniforms = sobol_engine.random_base2(m=int(np.log2(n_power)))[1 : n + 1]
    eps = np.finfo(float).eps
    uniforms = np.clip(uniforms, eps, 1.0 - eps)
    proposals = norm.ppf(uniforms)
    _SOBOL_STANDARD_NORMAL_CACHE[key] = proposals
    return proposals


def riemann_fast_missing_moments(
    *,
    mu_cond: np.ndarray,
    Sigma_cond: np.ndarray,
    lower: np.ndarray,
    upper: np.ndarray,
    sampler: SamplerConfig,
    show_progress: bool = True,
    progress_bar: Optional[object] = None,
) -> Tuple[np.ndarray, np.ndarray, float]:
    r = len(mu_cond)
    if sampler.fast_grid_points <= 0:
        raise ValueError("fast_grid_points must be positive for FAST self-censoring approximation.")
    if not np.all(np.isfinite(lower)) or not np.all(np.isfinite(upper)):
        raise ValueError("FAST self-censoring Riemann approximation requires finite interval bounds.")
    widths = upper - lower
    if np.any(widths <= 0.0):
        raise ValueError("FAST self-censoring Riemann approximation requires positive interval widths.")
    Sigma_cond = eigen_clip_spd(Sigma_cond, lower=1e-12)
    sign, logdet = np.linalg.slogdet(Sigma_cond)
    if sign <= 0 or not np.isfinite(logdet):
        raise ValueError("FAST self-censoring covariance block must be positive definite.")
    Sigma_inv = np.linalg.inv(Sigma_cond)
    grid_points = int(sampler.fast_grid_points)
    max_grid_size = int(sampler.fast_max_grid_size)
    grid_size = grid_points ** r
    if grid_size > max_grid_size:
        raise ValueError(
            f"FAST self-censoring Riemann grid size {grid_size} exceeds "
            f"fast_max_grid_size={max_grid_size}."
        )
    delta = widths / grid_points
    log_delta_volume = float(np.sum(np.log(delta)))
    axes = [lower[j] + (np.arange(grid_points, dtype=float) + 0.5) * delta[j] for j in range(r)]
    log_density_constant = -0.5 * (r * np.log(2.0 * np.pi) + logdet)
    log_weight_max = -np.inf
    scaled_weight_sum = 0.0
    scaled_N1 = np.zeros(r, dtype=float)
    scaled_N2 = np.zeros((r, r), dtype=float)

    close_grid_iter = False
    if progress_bar is not None:
        reset_progress_bar(progress_bar, total=grid_size, desc=f"FAST Riemann grid ({grid_points})")
        grid_iter = itertools.product(*axes)
    elif show_progress:
        grid_iter = tqdm(
            itertools.product(*axes),
            total=grid_size,
            desc=f"FAST Riemann grid ({grid_points})",
            unit="node",
            position=2,
            leave=False,
        )
        close_grid_iter = True
    else:
        grid_iter = itertools.product(*axes)

    try:
        for z_tuple in grid_iter:
            z = np.asarray(z_tuple, dtype=float)
            centered = z - mu_cond
            log_weight = log_density_constant - 0.5 * float(centered @ Sigma_inv @ centered) + log_delta_volume
            if not np.isfinite(log_weight):
                continue
            if log_weight > log_weight_max:
                scale = 0.0 if not np.isfinite(log_weight_max) else np.exp(log_weight_max - log_weight)
                scaled_weight_sum = scaled_weight_sum * scale + 1.0
                scaled_N1 = scaled_N1 * scale + z
                scaled_N2 = scaled_N2 * scale + np.outer(z, z)
                log_weight_max = log_weight
            else:
                scaled_weight = np.exp(log_weight - log_weight_max)
                scaled_weight_sum += scaled_weight
                scaled_N1 += scaled_weight * z
                scaled_N2 += scaled_weight * np.outer(z, z)
            if progress_bar is not None:
                progress_bar.update(1)
    finally:
        if close_grid_iter:
            close_progress_bar(grid_iter)

    if not np.isfinite(log_weight_max) or not np.isfinite(scaled_weight_sum) or scaled_weight_sum <= 0.0:
        raise ValueError("FAST self-censoring Riemann approximation found zero feasible probability.")
    log_D_hat = float(log_weight_max + np.log(scaled_weight_sum))
    z_mean = scaled_N1 / scaled_weight_sum
    Q_z = scaled_N2 / scaled_weight_sum
    log_p_hat = log_D_hat
    if not np.isfinite(log_p_hat):
        raise ValueError("FAST self-censoring Riemann approximation produced invalid log probability estimate.")
    return z_mean, Q_z, log_p_hat


def exponential_tilting_fast_missing_moments(
    *,
    mu_cond: np.ndarray,
    Sigma_cond: np.ndarray,
    H: np.ndarray,
    c: np.ndarray,
    sampler: SamplerConfig,
    rng: np.random.Generator,
    progress_bar: Optional[object] = None,
) -> Tuple[np.ndarray, np.ndarray, float]:
    r = len(mu_cond)
    if sampler.fast_tilt_samples <= 0:
        raise ValueError("fast_tilt_samples must be positive for FAST exponential-tilting approximation.")
    if sampler.fast_min_feasible <= 0:
        raise ValueError("fast_min_feasible must be positive for FAST exponential-tilting approximation.")
    if sampler.fast_min_feasible > sampler.fast_tilt_samples:
        raise ValueError("fast_min_feasible cannot exceed fast_tilt_samples.")

    Sigma_cond = eigen_clip_spd(Sigma_cond, lower=1e-12)
    L = np.linalg.cholesky(Sigma_cond)
    H_prime = H @ L
    c_prime = c - H @ mu_cond
    zero = np.zeros(r, dtype=float)

    try:
        lambda_shift = project_onto_linear_constraints(
            zero,
            H_prime,
            c_prime,
            error_prefix="FAST linear-censoring tilted projection",
        )
    except ValueError as exc:
        raise ValueError("FAST exponential-tilting approximation failed to project zero onto the tilted feasible domain.") from exc

    max_proposals = int(sampler.fast_tilt_samples)
    min_feasible = int(sampler.fast_min_feasible)
    if progress_bar is not None:
        reset_progress_bar(progress_bar, total=max_proposals, desc="FAST exponential tilting")

    standard_normal_proposals = sobol_standard_normals(r, max_proposals)
    total_drawn = 0
    feasible_count = 0
    feasible_chunks = []
    proposal_chunk_size = min(max_proposals, min_feasible)
    while total_drawn < max_proposals and feasible_count < min_feasible:
        draw_count = min(proposal_chunk_size, max_proposals - total_drawn)
        z_prime_batch = lambda_shift + standard_normal_proposals[total_drawn : total_drawn + draw_count]
        feasible = np.all(z_prime_batch @ H_prime.T <= c_prime[None, :] + 1e-10, axis=1)
        if np.any(feasible):
            feasible_batch = z_prime_batch[feasible]
            feasible_chunks.append(feasible_batch)
            feasible_count += feasible_batch.shape[0]
        total_drawn += draw_count
        if progress_bar is not None:
            progress_bar.update(draw_count)

    if progress_bar is not None:
        complete_progress_bar(progress_bar)

    if feasible_count < min_feasible:
        raise ValueError(
            f"FAST exponential-tilting approximation found only {feasible_count} feasible samples "
            f"after {total_drawn} proposals; requires at least {min_feasible}."
        )

    z_prime_feasible = np.vstack(feasible_chunks)
    log_weights = -z_prime_feasible @ lambda_shift + 0.5 * float(lambda_shift @ lambda_shift)
    log_weight_total = float(logsumexp(log_weights))
    if not np.isfinite(log_weight_total):
        raise ValueError("FAST exponential-tilting approximation produced zero total importance weight.")

    normalized_weights = np.exp(log_weights - log_weight_total)
    z_feasible = mu_cond + z_prime_feasible @ L.T
    z_mean = normalized_weights @ z_feasible
    Q_z = np.einsum("b,bi,bj->ij", normalized_weights, z_feasible, z_feasible)
    log_p_hat = log_weight_total - np.log(total_drawn)
    if not np.isfinite(log_p_hat):
        raise ValueError("FAST exponential-tilting approximation produced invalid log probability estimate.")
    return z_mean, Q_z, log_p_hat


def conditional_completed_moments_fast(
    *,
    X_obs_row: np.ndarray,
    mask_row: np.ndarray,
    nu: np.ndarray,
    T: np.ndarray,
    censoring_rule: CensoringRule,
    sampler: SamplerConfig,
    rng: np.random.Generator,
    show_progress: bool = True,
    progress_bar: Optional[object] = None,
) -> Tuple[np.ndarray, np.ndarray, float]:
    d = X_obs_row.shape[0]
    obs_idx, miss_idx = split_indices(mask_row)
    y = X_obs_row[obs_idx]
    r = len(miss_idx)

    if r == 0:
        x = X_obs_row.astype(float).copy()
        return x, np.outer(x, x), 0.0

    mu_cond, Sigma_cond, _, _ = conditional_gaussian_from_canonical(nu, T, obs_idx, miss_idx, y)
    if isinstance(censoring_rule, SelfCensoringRule):
        lower = censoring_rule.intervals[miss_idx, 0]
        upper = censoring_rule.intervals[miss_idx, 1]
        try:
            z_mean, Q_z, log_p_hat = riemann_fast_missing_moments(
                mu_cond=mu_cond,
                Sigma_cond=Sigma_cond,
                lower=lower,
                upper=upper,
                sampler=sampler,
                show_progress=show_progress,
                progress_bar=progress_bar,
            )
        except ValueError as riemann_error:
            H_box = np.vstack([np.eye(r, dtype=float), -np.eye(r, dtype=float)])
            c_box = np.concatenate([upper, -lower])
            try:
                z_mean, Q_z, log_p_hat = exponential_tilting_fast_missing_moments(
                    mu_cond=mu_cond,
                    Sigma_cond=Sigma_cond,
                    H=H_box,
                    c=c_box,
                    sampler=sampler,
                    rng=rng,
                    progress_bar=progress_bar,
                )
            except ValueError as tilt_error:
                raise ValueError(
                    "FAST self-censoring approximation failed: "
                    f"Riemann failed ({riemann_error}); exponential tilting failed ({tilt_error})."
                ) from tilt_error
    elif isinstance(censoring_rule, LinearCensoringRule):
        H, c = censoring_rule.constraints(obs_idx, miss_idx, y, mask_row)
        z_mean, Q_z, log_p_hat = exponential_tilting_fast_missing_moments(
            mu_cond=mu_cond,
            Sigma_cond=Sigma_cond,
            H=H,
            c=c,
            sampler=sampler,
            rng=rng,
            progress_bar=progress_bar,
        )
    else:
        raise TypeError("FAST approximation supports SelfCensoringRule and LinearCensoringRule only.")

    x_bar, Xi = completed_moments_from_missing(y, obs_idx, miss_idx, z_mean, Q_z, d)
    return x_bar, Xi, log_p_hat



## Projection Steps

For PSGD, projection is applied in the proposal's A1, A2, A3 order. For EM raw covariance updates, only symmetric positive-definite covariance repair is applied.


In [10]:

def project_precision_A1(T_plus: np.ndarray, projection: ProjectionConfig) -> np.ndarray:
    return eigen_clip_spd(T_plus, lower=projection.tau_lower, upper=projection.tau_upper)


def project_precision_A2(T_1: np.ndarray, T_ref: np.ndarray, projection: ProjectionConfig) -> np.ndarray:
    if not projection.use_A2:
        return T_1
    if projection.r_T is None or not np.isfinite(projection.r_T):
        return T_1
    Delta = T_1 - T_ref
    d_T = np.linalg.norm(Delta, ord="fro")
    if d_T <= projection.r_T or d_T == 0:
        return T_1
    return symmetrize(T_ref + (projection.r_T / d_T) * Delta)


def project_nu_A3(nu_plus: np.ndarray, projection: ProjectionConfig) -> np.ndarray:
    if not projection.use_A3:
        return nu_plus
    if projection.r_nu is None or not np.isfinite(projection.r_nu):
        return nu_plus
    d_nu = np.linalg.norm(nu_plus)
    if d_nu <= projection.r_nu or d_nu == 0:
        return nu_plus
    return (projection.r_nu / d_nu) * nu_plus


def project_sgd_parameters(
    nu_plus: np.ndarray,
    T_plus: np.ndarray,
    T_ref: np.ndarray,
    projection: ProjectionConfig,
    fixed_T: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    if fixed_T is None:
        T_1 = project_precision_A1(T_plus, projection)
        T_2 = project_precision_A2(T_1, T_ref, projection)
    else:
        T_2 = fixed_T.copy()
    nu_3 = project_nu_A3(nu_plus, projection)
    return nu_3, T_2


def repair_raw_covariance(Sigma: np.ndarray, projection: ProjectionConfig) -> np.ndarray:
    return eigen_clip_spd(Sigma, lower=projection.covariance_eig_floor, upper=None)



## Parameter Update Modules


In [11]:

def gradient_signal_to_noise(gradients: np.ndarray, epsilon_num: float) -> Tuple[np.ndarray, float]:
    gradients = np.asarray(gradients, dtype=float)
    B = gradients.shape[0]
    g_hat = np.mean(gradients, axis=0)
    centered = gradients.reshape(B, -1) - g_hat.reshape(1, -1)
    s2_hat = np.sum(centered * centered) / (B - 1) if B > 1 else 0.0
    se_hat = np.sqrt(max(s2_hat, 0.0) / B)
    snr = np.linalg.norm(g_hat) / (se_hat + epsilon_num)
    return g_hat, float(snr)


def sgd_gradient_diagnostics(
    *,
    nu: np.ndarray,
    T: np.ndarray,
    x_bars: np.ndarray,
    Xi_bars: np.ndarray,
    epsilon_num: float,
    fixed_covariance: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, Optional[np.ndarray], float, float]:
    Sigma_current = np.linalg.inv(T)
    mu_current = Sigma_current @ nu

    g_nu_records = mu_current[None, :] - x_bars
    g_nu, snr_nu = gradient_signal_to_noise(g_nu_records, epsilon_num)

    if fixed_covariance is not None:
        return g_nu, None, snr_nu, np.nan

    centered_second_moment = Sigma_current + np.outer(mu_current, mu_current)
    G_T_records = 0.5 * (Xi_bars - centered_second_moment)
    G_T, snr_T = gradient_signal_to_noise(G_T_records, epsilon_num)
    return g_nu, G_T, snr_nu, snr_T


def clipped_snr_learning_rate_scale(snr: float, lower: float = 0.1, upper: float = 10.0) -> float:
    try:
        snr = float(snr)
    except (TypeError, ValueError):
        return lower
    if np.isnan(snr):
        return lower
    return float(np.clip(snr, lower, upper))


def sgd_update(
    *,
    nu: np.ndarray,
    T: np.ndarray,
    x_bars: np.ndarray,
    Xi_bars: np.ndarray,
    t: int,
    optimizer: OptimizerConfig,
    projection: ProjectionConfig,
    T_ref: np.ndarray,
    fixed_covariance: Optional[np.ndarray] = None,
    g_nu: Optional[np.ndarray] = None,
    G_T: Optional[np.ndarray] = None,
    snr_nu: float = 1.0,
    snr_T: float = 1.0,
) -> Tuple[np.ndarray, np.ndarray]:
    Sigma_current = np.linalg.inv(T)
    mu_current = Sigma_current @ nu

    if g_nu is None:
        x_mean = np.mean(x_bars, axis=0)
        g_nu = mu_current - x_mean

    lr_decay_factor = 1.0 / t if optimizer.decay_lr else 1.0
    snr_nu_scale = clipped_snr_learning_rate_scale(snr_nu)
    eta_nu_t = optimizer.eta_nu0 * snr_nu_scale * lr_decay_factor
    nu_plus = nu - eta_nu_t * g_nu

    fixed_T = None
    T_plus = T.copy()
    if fixed_covariance is not None:
        fixed_T = np.linalg.inv(fixed_covariance)
    else:
        if G_T is None:
            Xi_mean = np.mean(Xi_bars, axis=0)
            G_T = 0.5 * (Xi_mean - Sigma_current - np.outer(mu_current, mu_current))
        snr_T_scale = clipped_snr_learning_rate_scale(snr_T)
        eta_T_t = optimizer.eta_T0 * snr_T_scale * lr_decay_factor
        T_plus = T - eta_T_t * G_T

    return project_sgd_parameters(nu_plus, T_plus, T_ref, projection, fixed_T=fixed_T)


def em_common_learning_rate(t: int, optimizer: OptimizerConfig) -> float:
    lr_decay_factor = 1.0 / t if optimizer.decay_lr else 1.0
    eta_nu_base_t = optimizer.eta_nu0 * lr_decay_factor
    eta_T_base_t = optimizer.eta_T0 * lr_decay_factor
    gamma_t = 0.5 * (eta_nu_base_t + eta_T_base_t)
    return min(1.0, gamma_t)


def em_update(
    *,
    x_bars: np.ndarray,
    Xi_bars: np.ndarray,
    S_x_prev: np.ndarray,
    S_xx_prev: np.ndarray,
    t: int,
    optimizer: OptimizerConfig,
    projection: ProjectionConfig,
    fixed_covariance: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    gamma_t = em_common_learning_rate(t, optimizer)
    gamma_nu_t = gamma_t
    gamma_T_t = gamma_t
    batch_x = np.mean(x_bars, axis=0)
    batch_xx = np.mean(Xi_bars, axis=0)
    S_x = (1.0 - gamma_nu_t) * S_x_prev + gamma_nu_t * batch_x
    S_xx = (1.0 - gamma_T_t) * S_xx_prev + gamma_T_t * batch_xx

    mu_raw = S_x
    if fixed_covariance is not None:
        return mu_raw, symmetrize(fixed_covariance), S_x, S_xx

    Sigma_raw = S_xx - np.outer(S_x, S_x)
    Sigma = repair_raw_covariance(Sigma_raw, projection)
    return mu_raw, Sigma, S_x, S_xx



## Initialization

The initial completed matrix fills each missing block with a feasible point from the censoring rule, then applies the proposal's raw Gaussian initializer and canonical conversion.


In [12]:

def initial_completed_matrix(
    X_obs: np.ndarray,
    mask: np.ndarray,
    censoring_rule: CensoringRule,
) -> np.ndarray:
    if np.all(mask):
        return X_obs.astype(float).copy()

    n, d = X_obs.shape
    column_medians = np.zeros(d, dtype=float)
    for j in range(d):
        observed_column = X_obs[mask[:, j], j]
        if observed_column.size > 0:
            column_medians[j] = float(np.median(observed_column))
    X0 = np.empty((n, d), dtype=float)
    for i in range(n):
        obs_idx, miss_idx = split_indices(mask[i])
        if len(miss_idx) == 0:
            X0[i] = X_obs[i].astype(float)
            continue
        y = X_obs[i, obs_idx]
        z_proposed = column_medians[miss_idx]
        z0 = censoring_rule.project(z_proposed, obs_idx, miss_idx, y, mask[i])
        X0[i] = reconstruct_full(y, z0, obs_idx, miss_idx, d)
    return X0


def initialize_parameters(
    X_obs: np.ndarray,
    mask: np.ndarray,
    censoring_rule: CensoringRule,
    projection: ProjectionConfig,
    fixed_covariance: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    X0 = initial_completed_matrix(X_obs, mask, censoring_rule)
    mu0 = np.mean(X0, axis=0)

    if fixed_covariance is not None:
        Sigma0 = symmetrize(fixed_covariance)
    else:
        centered = X0 - mu0
        denom = max(X0.shape[0] - 1, 1)
        Sigma0 = centered.T @ centered / denom
        Sigma0 = repair_raw_covariance(Sigma0, projection)

    nu0, T0 = raw_to_canonical(mu0, Sigma0)
    if fixed_covariance is None:
        T0 = project_precision_A1(T0, projection)
        Sigma0 = np.linalg.inv(T0)
        nu0 = T0 @ mu0
    return mu0, symmetrize(Sigma0), nu0, T0



## Main Model

This is the single entrypoint for the first-stage model. It returns predicted raw Gaussian parameters plus the diagnostic history.


In [13]:

def normalize_update_style(update_style: str) -> str:
    style = update_style.upper()
    if style not in {"SGD", "EM"}:
        raise ValueError("update_style must be strictly either 'SGD' or 'EM'.")
    return style


def normalize_approximation_style(approximation_style: str) -> str:
    style = approximation_style.upper()
    if style not in {"SLOW", "FAST", "AUTO"}:
        raise ValueError("approximation_style must be 'SLOW', 'FAST', or 'AUTO'.")
    return style


def default_precision_radius(n: int, d: int, tau_upper: float) -> float:
    numerator = (((n - d) * d) + ((n - d - 2) * d**2)) * tau_upper**2
    denominator = (n - d - 1) * (n - d - 4)
    if denominator <= 0:
        raise ValueError(
            "The default r_T formula requires (n - d - 1)(n - d - 4) > 0; "
            "pass ProjectionConfig(r_T=...) explicitly for this dataset size."
        )
    return float(numerator / denominator)


def grouped_observed_values_by_mask(X_obs: np.ndarray, mask: np.ndarray) -> Tuple[Dict[Tuple[int, ...], np.ndarray], Dict[Tuple[int, ...], int]]:
    groups: Dict[Tuple[int, ...], list] = {}
    counts: Dict[Tuple[int, ...], int] = {}
    for row, mask_row in zip(X_obs, mask):
        key = tuple(mask_row.astype(int).tolist())
        counts[key] = counts.get(key, 0) + 1
        y = row[mask_row]
        if len(y) > 0 and not np.all(np.isfinite(y)):
            raise ValueError("Observed validation values must be finite.")
        groups.setdefault(key, []).append(np.asarray(y, dtype=float))

    stacked_groups = {}
    for key, values in groups.items():
        r = int(sum(key))
        if r == 0:
            stacked_groups[key] = np.empty((len(values), 0), dtype=float)
        else:
            stacked_groups[key] = np.vstack(values)
    return stacked_groups, counts


def gaussian_kernel_matrix(X_left: np.ndarray, X_right: np.ndarray, H: np.ndarray) -> np.ndarray:
    r = X_left.shape[1]
    sign, logdet = np.linalg.slogdet(H)
    if sign <= 0 or not np.isfinite(logdet):
        raise ValueError("Kernel bandwidth matrix must be positive definite.")
    H_inv = np.linalg.inv(H)
    diff = X_left[:, None, :] - X_right[None, :, :]
    quadratic = np.einsum("...i,ij,...j->...", diff, H_inv, diff)
    log_kernel = -0.5 * (r * np.log(2.0 * np.pi) + logdet + quadratic)
    return np.exp(log_kernel)


def observed_validation_jeffreys(
    *,
    X_val: np.ndarray,
    mask_val: np.ndarray,
    mu: np.ndarray,
    Sigma: np.ndarray,
    censoring_rule: CensoringRule,
    rng: np.random.Generator,
    eta: float,
    kernel_epsilon: float,
    kernel_eig_floor: float,
) -> Dict[str, float]:
    n_val, d = X_val.shape
    if n_val <= 0:
        return {
            "observed_jeffreys": np.nan,
            "mask_jeffreys": np.nan,
            "value_jeffreys": np.nan,
            "valid_mask_count": 0,
        }

    Sigma = eigen_clip_spd(Sigma, lower=kernel_eig_floor)
    X_sim_full = rng.multivariate_normal(np.asarray(mu, dtype=float), Sigma, size=n_val)
    X_sim_obs, mask_sim = apply_censoring_rule(X_sim_full, censoring_rule)

    val_groups, val_counts = grouped_observed_values_by_mask(X_val, mask_val)
    sim_groups, sim_counts = grouped_observed_values_by_mask(X_sim_obs, mask_sim)
    mask_keys = set(val_counts) | set(sim_counts)
    mask_space_size = float(2**d)
    val_denominator = n_val + eta * mask_space_size
    sim_denominator = n_val + eta * mask_space_size

    mask_jeffreys = 0.0
    value_jeffreys = 0.0
    valid_mask_count = 0
    for key in mask_keys:
        n_val_m = val_counts.get(key, 0)
        n_sim_m = sim_counts.get(key, 0)
        p_tilde = (n_val_m + eta) / val_denominator
        q_tilde = (n_sim_m + eta) / sim_denominator
        mask_jeffreys += p_tilde * np.log(p_tilde / q_tilde) + q_tilde * np.log(q_tilde / p_tilde)

        r_m = int(sum(key))
        if r_m == 0 or n_val_m < 2 or n_sim_m < 2:
            continue
        Y_val = val_groups[key]
        Y_sim = sim_groups[key]
        N_m = n_val_m + n_sim_m
        Y_combined = np.vstack([Y_val, Y_sim])
        centered = Y_combined - np.mean(Y_combined, axis=0)
        S_m = centered.T @ centered / (N_m - 1)
        h_m = (4.0 / (r_m + 2.0)) ** (1.0 / (r_m + 4.0)) * (N_m ** (-1.0 / (r_m + 4.0)))
        H_m = eigen_clip_spd((h_m**2) * S_m, lower=kernel_eig_floor)

        K_val_val = gaussian_kernel_matrix(Y_val, Y_val, H_m)
        p_val_loo = (np.sum(K_val_val, axis=1) - np.diag(K_val_val)) / (n_val_m - 1)
        q_sim_at_val = np.mean(gaussian_kernel_matrix(Y_val, Y_sim, H_m), axis=1)
        kl_val_to_sim = float(np.mean(np.log((p_val_loo + kernel_epsilon) / (q_sim_at_val + kernel_epsilon))))

        K_sim_sim = gaussian_kernel_matrix(Y_sim, Y_sim, H_m)
        q_sim_loo = (np.sum(K_sim_sim, axis=1) - np.diag(K_sim_sim)) / (n_sim_m - 1)
        p_val_at_sim = np.mean(gaussian_kernel_matrix(Y_sim, Y_val, H_m), axis=1)
        kl_sim_to_val = float(np.mean(np.log((q_sim_loo + kernel_epsilon) / (p_val_at_sim + kernel_epsilon))))

        value_jeffreys += p_tilde * kl_val_to_sim + q_tilde * kl_sim_to_val
        valid_mask_count += 1

    observed_jeffreys = mask_jeffreys + value_jeffreys
    return {
        "observed_jeffreys": float(observed_jeffreys),
        "mask_jeffreys": float(mask_jeffreys),
        "value_jeffreys": float(value_jeffreys),
        "valid_mask_count": int(valid_mask_count),
    }


def validation_objective_value(
    *,
    X_val: np.ndarray,
    mask_val: np.ndarray,
    nu: np.ndarray,
    T: np.ndarray,
    censoring_rule: CensoringRule,
    sampler: SamplerConfig,
    rng: np.random.Generator,
) -> float:
    if len(X_val) == 0:
        return np.nan

    objective_mu, objective_Sigma = canonical_to_raw(nu, T)
    objective_terms = []
    for row, mask_row in zip(X_val, mask_val):
        obs_idx, miss_idx = split_indices(mask_row)
        if len(miss_idx) == 0:
            log_p_hat = 0.0
        else:
            try:
                _, _, log_p_hat = conditional_completed_moments_fast(
                    X_obs_row=row,
                    mask_row=mask_row,
                    nu=nu,
                    T=T,
                    censoring_rule=censoring_rule,
                    sampler=sampler,
                    rng=rng,
                    show_progress=False,
                    progress_bar=None,
                )
            except ValueError:
                continue

        if not np.isfinite(log_p_hat):
            continue

        y = row[obs_idx]
        observed_nll = gaussian_observed_negative_log_density(
            y=y,
            mu_obs=objective_mu[obs_idx],
            Sigma_obs=objective_Sigma[np.ix_(obs_idx, obs_idx)],
        )
        objective_contribution = observed_nll - log_p_hat
        if np.isfinite(objective_contribution):
            objective_terms.append(objective_contribution)

    return float(np.mean(objective_terms)) if objective_terms else np.nan



def validation_count_for_size(n: int, validation_fraction: float) -> int:
    n = int(n)
    validation_fraction = float(validation_fraction)
    if n <= 1 or validation_fraction <= 0.0:
        return 0
    return int(min(n - 1, max(1, np.floor(validation_fraction * n))))


def prefix_validation_indices(n: int, validation_fraction: float) -> np.ndarray:
    n_val = validation_count_for_size(n, validation_fraction)
    return np.arange(n_val, dtype=int)


def normalize_validation_indices(validation_indices: ArrayLike, n: int) -> np.ndarray:
    indices = np.asarray(validation_indices, dtype=int).reshape(-1)
    if indices.size == 0:
        return indices
    if np.any(indices < 0) or np.any(indices >= n):
        raise ValueError("validation_indices must be valid row indices for X_obs.")
    if len(np.unique(indices)) != len(indices):
        raise ValueError("validation_indices must not contain duplicates.")
    if len(indices) >= n:
        raise ValueError("validation_indices must leave at least one training row.")
    return indices


def fit_mnar_gaussian(
    X_obs: ArrayLike,
    true_mean: Optional[ArrayLike] = None,
    true_covariance: Optional[ArrayLike] = None,
    update_style: str = "SGD",
    approximation_style: str = "AUTO",
    *,
    censoring_rule: CensoringRule,
    mask: Optional[ArrayLike] = None,
    known_covariance: bool = False,
    optimizer: Optional[OptimizerConfig] = None,
    sampler: Optional[SamplerConfig] = None,
    projection: Optional[ProjectionConfig] = None,
    random_state: Optional[int] = None,
    validation_indices: Optional[ArrayLike] = None,
    show_progress: bool = True,
    progress_position_offset: int = 0,
    progress_dashboard: Optional[object] = None,
) -> FitResult:
    X_obs = np.asarray(X_obs, dtype=float)
    mask = infer_mask(X_obs, mask)

    if X_obs.ndim != 2:
        raise ValueError("X_obs must be a 2D matrix of shape (n, d).")
    n, d = X_obs.shape
    if mask.shape != X_obs.shape:
        raise ValueError("mask must have the same shape as X_obs.")
    if not isinstance(known_covariance, (bool, np.bool_)):
        raise TypeError("known_covariance must be a bool.")
    known_covariance = bool(known_covariance)

    true_mean_arr = None if true_mean is None else np.asarray(true_mean, dtype=float)
    true_covariance_arr = None if true_covariance is None else symmetrize(true_covariance)
    if true_mean_arr is not None and true_mean_arr.shape != (d,):
        raise ValueError("true_mean must have shape (d,) when supplied.")
    if true_covariance_arr is not None and true_covariance_arr.shape != (d, d):
        raise ValueError("true_covariance must have shape (d, d) when supplied.")
    if known_covariance and true_covariance_arr is None:
        raise ValueError("known_covariance=True requires true_covariance, which is then fixed during fitting.")

    update_style = normalize_update_style(update_style)
    approximation_style = normalize_approximation_style(approximation_style)
    optimizer = optimizer or OptimizerConfig()
    sampler = sampler or SamplerConfig()
    projection = projection or ProjectionConfig()
    rng = np.random.default_rng(random_state)
    objective_rng = np.random.default_rng(
        None if random_state is None else np.random.SeedSequence([int(random_state), 1])
    )
    if not isinstance(progress_position_offset, (int, np.integer)) or progress_position_offset < 0:
        raise ValueError("progress_position_offset must be a non-negative integer.")
    progress_position_offset = int(progress_position_offset)
    has_missing = not np.all(mask)
    uses_slow = has_missing and approximation_style in {"SLOW", "AUTO"}
    uses_fast = has_missing and approximation_style in {"FAST", "AUTO"}

    if optimizer.n_iter <= 0:
        raise ValueError("optimizer.n_iter must be positive.")
    if optimizer.batch_size <= 0:
        raise ValueError("optimizer.batch_size must be positive.")
    if not np.isfinite(optimizer.eta_nu0) or optimizer.eta_nu0 <= 0.0:
        raise ValueError("eta_nu0 must be positive and finite.")
    if not np.isfinite(optimizer.eta_T0) or optimizer.eta_T0 < 0.0:
        raise ValueError("eta_T0 must be non-negative and finite.")
    if not isinstance(optimizer.decay_lr, (bool, np.bool_)):
        raise TypeError("decay_lr must be a bool.")
    if not np.isfinite(optimizer.snr_epsilon) or optimizer.snr_epsilon <= 0.0:
        raise ValueError("snr_epsilon must be positive and finite.")
    if not isinstance(optimizer.live_plot, (bool, np.bool_)):
        raise TypeError("live_plot must be a bool.")
    if not isinstance(optimizer.live_plot_every, (int, np.integer)) or optimizer.live_plot_every <= 0:
        raise ValueError("live_plot_every must be a positive integer.")
    if not np.isfinite(optimizer.validation_fraction) or not (0.0 <= optimizer.validation_fraction < 1.0):
        raise ValueError("validation_fraction must be in [0, 1).")
    if not isinstance(optimizer.validation_every, (int, np.integer)) or optimizer.validation_every <= 0:
        raise ValueError("validation_every must be a positive integer.")
    if optimizer.validation_impatience_threshold is not None:
        if not isinstance(optimizer.validation_impatience_threshold, (int, np.integer)) or optimizer.validation_impatience_threshold < 0:
            raise ValueError("validation_impatience_threshold must be a non-negative integer, or None to disable validation early stopping.")
    if not isinstance(optimizer.track_objective_impatience, (bool, np.bool_)):
        raise TypeError("track_objective_impatience must be a bool.")
    if not isinstance(optimizer.track_JD_impatience, (bool, np.bool_)):
        raise TypeError("track_JD_impatience must be a bool.")
    if not isinstance(optimizer.compute_validation_observed_jd, (bool, np.bool_)):
        raise TypeError("compute_validation_observed_jd must be a bool.")
    if not np.isfinite(optimizer.validation_mask_eta) or optimizer.validation_mask_eta <= 0.0:
        raise ValueError("validation_mask_eta must be positive and finite.")
    if not np.isfinite(optimizer.validation_kernel_epsilon) or optimizer.validation_kernel_epsilon <= 0.0:
        raise ValueError("validation_kernel_epsilon must be positive and finite.")
    if not np.isfinite(optimizer.validation_kernel_eig_floor) or optimizer.validation_kernel_eig_floor <= 0.0:
        raise ValueError("validation_kernel_eig_floor must be positive and finite.")
    if uses_slow and (sampler.K <= 0 or sampler.thin <= 0 or sampler.burn_in < 0):
        raise ValueError("SLOW sampler requires K > 0, thin > 0, and burn_in >= 0.")
    if uses_fast and (
        sampler.fast_grid_points <= 0
        or sampler.fast_max_grid_size <= 0
        or sampler.fast_tilt_samples <= 0
        or sampler.fast_min_feasible <= 0
    ):
        raise ValueError("FAST sampler requires fast_grid_points > 0, fast_max_grid_size > 0, fast_tilt_samples > 0, and fast_min_feasible > 0.")
    if uses_fast and sampler.fast_min_feasible > sampler.fast_tilt_samples:
        raise ValueError("fast_min_feasible cannot exceed fast_tilt_samples.")

    original_n = n
    X_val = np.empty((0, d), dtype=float)
    mask_val = np.empty((0, d), dtype=bool)
    val_idx = np.empty(0, dtype=int)
    train_idx = np.arange(n, dtype=int)
    validation_split_strategy = "none"
    if validation_indices is not None:
        val_idx = normalize_validation_indices(validation_indices, n)
        validation_split_strategy = "explicit"
    elif optimizer.validation_fraction > 0.0 and n > 1:
        n_val = validation_count_for_size(n, optimizer.validation_fraction)
        split_permutation = rng.permutation(n)
        val_idx = split_permutation[:n_val]
        validation_split_strategy = "random"

    if len(val_idx) > 0:
        train_mask = np.ones(n, dtype=bool)
        train_mask[val_idx] = False
        train_idx = np.flatnonzero(train_mask)
        X_val = X_obs[val_idx].copy()
        mask_val = mask[val_idx].copy()
        X_obs = X_obs[train_idx].copy()
        mask = mask[train_idx].copy()
        n = X_obs.shape[0]
    validation_enabled = len(X_val) > 0

    fixed_covariance = true_covariance_arr.copy() if known_covariance else None
    mu, Sigma, nu, T = initialize_parameters(X_obs, mask, censoring_rule, projection, fixed_covariance)
    T_ref = T.copy()
    projection = replace(
        projection,
        r_nu=(projection.r_nu if projection.r_nu is not None else 2.0 * max(np.linalg.norm(nu), 1.0)),
        r_T=(projection.r_T if projection.r_T is not None else default_precision_radius(n, d, projection.tau_upper)),
    )

    bar_nu = nu.copy()
    bar_T = T.copy()
    S_x = mu.copy()
    S_xx = Sigma + np.outer(mu, mu)
    warm_states: Dict[int, np.ndarray] = {}
    compute_error = true_mean_arr is not None and true_covariance_arr is not None
    history = {"l2_error": [], "mean_squared_l2_error": []} if compute_error else {}
    history["validation_split"] = {
        "strategy": validation_split_strategy,
        "n_total": int(original_n),
        "n_train": int(n),
        "n_validation": int(len(val_idx)),
        "validation_indices_are_prefix": bool(np.array_equal(val_idx, np.arange(len(val_idx), dtype=int))),
        "validation_prefix_stop": int(len(val_idx)) if np.array_equal(val_idx, np.arange(len(val_idx), dtype=int)) else None,
        "validation_indices_preview": val_idx[:20].astype(int).tolist(),
    }
    history["theta"] = []
    history["batch_size"] = []
    history["objective"] = []
    history["objective_iteration"] = []
    history["objective_impatience"] = []
    history["validation_iteration"] = []
    history["validation_observed_jeffreys"] = []
    history["validation_mask_jeffreys"] = []
    history["validation_value_jeffreys"] = []
    history["validation_valid_mask_count"] = []
    history["validation_impatience"] = []
    if update_style == "EM":
        history["gamma"] = []
        history["gamma_nu"] = []
        history["gamma_T"] = []
    if update_style == "SGD":
        history["snr_nu"] = []
        history["snr_T"] = []
    if approximation_style == "AUTO":
        history["fast_slow_fallback_count"] = []
    objective_minimum = np.inf
    objective_impatience = 0
    validation_impatience = 0
    history["theta"].append(parameter_vector(mu, Sigma))
    if compute_error:
        if update_style == "SGD":
            l2_0, mean_sq_l2_0 = parameter_error(
                update_style="SGD",
                known_covariance=known_covariance,
                true_mean=true_mean_arr,
                true_covariance=true_covariance_arr,
                nu_est=bar_nu,
                T_est=bar_T,
            )
        else:
            l2_0, mean_sq_l2_0 = parameter_error(
                update_style="EM",
                known_covariance=known_covariance,
                true_mean=true_mean_arr,
                true_covariance=true_covariance_arr,
                mu_est=mu,
                Sigma_est=Sigma,
            )
        history["l2_error"].append(l2_0)
        history["mean_squared_l2_error"].append(mean_sq_l2_0)

    if validation_enabled:
        objective_value_0 = validation_objective_value(
            X_val=X_val,
            mask_val=mask_val,
            nu=bar_nu if update_style == "SGD" else nu,
            T=bar_T if update_style == "SGD" else T,
            censoring_rule=censoring_rule,
            sampler=sampler,
            rng=objective_rng,
        )
        history["objective_iteration"].append(0)
        history["objective"].append(objective_value_0)
        if optimizer.track_objective_impatience and np.isfinite(objective_value_0):
            objective_minimum = objective_value_0
        history["objective_impatience"].append(objective_impatience)

    dashboard_active = show_progress and progress_dashboard is not None
    dashboard_iteration_bar = None
    if dashboard_active:
        dashboard_iteration_bar = progress_dashboard.bar("iterations")
        reset_progress_bar(dashboard_iteration_bar, total=optimizer.n_iter, desc="Optimizer iterations")
        iteration_range = range(1, optimizer.n_iter + 1)
    else:
        iteration_range = tqdm(
            range(1, optimizer.n_iter + 1),
            desc="Optimizer iterations",
            unit="iter",
            position=progress_position_offset,
            leave=False,
            disable=not show_progress,
        )
    live_plot_handle = None
    if optimizer.live_plot and (compute_error or validation_enabled):
        if dashboard_active and hasattr(progress_dashboard, "live_plot_handle"):
            live_plot_handle = progress_dashboard.live_plot_handle()
        else:
            live_plot_handle = display(HTML(""), display_id=True)

    try:
        for t in iteration_range:
            current_batch_size = max(1, int(np.ceil(optimizer.batch_size * expit(t))))
            batch_idx = rng.integers(0, n, size=current_batch_size)

            x_bars = []
            Xi_bars = []
            fast_slow_fallback_count = 0

            for i in batch_idx:
                use_slow = approximation_style == "SLOW"
                if not use_slow:
                    try:
                        x_bar, Xi_bar, _log_p_hat = conditional_completed_moments_fast(
                            X_obs_row=X_obs[i],
                            mask_row=mask[i],
                            nu=nu,
                            T=T,
                            censoring_rule=censoring_rule,
                            sampler=sampler,
                            rng=rng,
                            show_progress=False,
                            progress_bar=None,
                        )
                    except ValueError:
                        if approximation_style == "FAST":
                            raise
                        fast_slow_fallback_count += 1
                        use_slow = True

                if use_slow:
                    warm_state = warm_states.get(int(i)) if sampler.warm_start else None
                    x_bar, Xi_bar, final_state = conditional_completed_moments_slow(
                        X_obs_row=X_obs[i],
                        mask_row=mask[i],
                        nu=nu,
                        T=T,
                        censoring_rule=censoring_rule,
                        sampler=sampler,
                        rng=rng,
                        warm_state=warm_state,
                        show_progress=False,
                        progress_bar=None,
                    )
                    if sampler.warm_start:
                        warm_states[int(i)] = final_state

                x_bars.append(x_bar)
                Xi_bars.append(Xi_bar)
            x_bars = np.asarray(x_bars, dtype=float)
            Xi_bars = np.asarray(Xi_bars, dtype=float)
            stop_after_iteration = False
            g_nu = None
            G_T = None
            snr_nu = np.nan
            snr_T = np.nan
            history["batch_size"].append(int(len(batch_idx)))
            if update_style == "SGD":
                g_nu, G_T, snr_nu, snr_T = sgd_gradient_diagnostics(
                    nu=nu,
                    T=T,
                    x_bars=x_bars,
                    Xi_bars=Xi_bars,
                    epsilon_num=optimizer.snr_epsilon,
                    fixed_covariance=fixed_covariance,
                )
                history["snr_nu"].append(snr_nu)
                history["snr_T"].append(snr_T)
            if approximation_style == "AUTO":
                history["fast_slow_fallback_count"].append(fast_slow_fallback_count)

            if update_style == "SGD":
                nu, T = sgd_update(
                    nu=nu,
                    T=T,
                    x_bars=x_bars,
                    Xi_bars=Xi_bars,
                    t=t,
                    optimizer=optimizer,
                    projection=projection,
                    T_ref=T_ref,
                    fixed_covariance=fixed_covariance,
                    g_nu=g_nu,
                    G_T=G_T,
                    snr_nu=snr_nu,
                    snr_T=snr_T,
                )
                bar_nu = ((t - 1) / t) * bar_nu + (1.0 / t) * nu
                bar_T = ((t - 1) / t) * bar_T + (1.0 / t) * T
                mu_est, Sigma_est = canonical_to_raw(bar_nu, bar_T)
                if compute_error:
                    l2, mean_sq_l2 = parameter_error(
                        update_style="SGD",
                        known_covariance=known_covariance,
                        true_mean=true_mean_arr,
                        true_covariance=true_covariance_arr,
                        nu_est=bar_nu,
                        T_est=bar_T,
                    )
            else:
                gamma_t = em_common_learning_rate(t, optimizer)
                gamma_nu_t = gamma_t
                gamma_T_t = gamma_t
                mu, Sigma, S_x, S_xx = em_update(
                    x_bars=x_bars,
                    Xi_bars=Xi_bars,
                    S_x_prev=S_x,
                    S_xx_prev=S_xx,
                    t=t,
                    optimizer=optimizer,
                    projection=projection,
                    fixed_covariance=fixed_covariance,
                )
                history["gamma"].append(gamma_nu_t)
                history["gamma_nu"].append(gamma_nu_t)
                history["gamma_T"].append(gamma_T_t)
                nu, T = raw_to_canonical(mu, Sigma)
                mu_est, Sigma_est = mu.copy(), Sigma.copy()
                if compute_error:
                    l2, mean_sq_l2 = parameter_error(
                        update_style="EM",
                        known_covariance=known_covariance,
                        true_mean=true_mean_arr,
                        true_covariance=true_covariance_arr,
                        mu_est=mu_est,
                        Sigma_est=Sigma_est,
                    )

            history["theta"].append(parameter_vector(mu_est, Sigma_est))
            postfix = {}
            if compute_error:
                history["l2_error"].append(l2)
                history["mean_squared_l2_error"].append(mean_sq_l2)
                delta_error = np.nan if len(history["mean_squared_l2_error"]) < 2 else history["mean_squared_l2_error"][-1] - history["mean_squared_l2_error"][-2]
                postfix["Δerror"] = f"{delta_error:+.6g}"
            validation_recorded = False

            if validation_enabled and (t % optimizer.validation_every == 0 or t == optimizer.n_iter):
                objective_nu, objective_T = (bar_nu, bar_T) if update_style == "SGD" else (nu, T)
                objective_value = validation_objective_value(
                    X_val=X_val,
                    mask_val=mask_val,
                    nu=objective_nu,
                    T=objective_T,
                    censoring_rule=censoring_rule,
                    sampler=sampler,
                    rng=objective_rng,
                )
                history["objective_iteration"].append(int(t))
                history["objective"].append(objective_value)
                validation_recorded = True
                delta_objective = np.nan if len(history["objective"]) < 2 else history["objective"][-1] - history["objective"][-2]
                postfix["Δobjective"] = f"{delta_objective:+.6g}"
                if optimizer.track_objective_impatience and np.isfinite(objective_value):
                    if objective_value < objective_minimum:
                        objective_minimum = objective_value
                        objective_impatience = 0
                    elif objective_value > objective_minimum:
                        objective_impatience += 1
                    if optimizer.validation_impatience_threshold is not None and objective_impatience > optimizer.validation_impatience_threshold:
                        stop_after_iteration = True
                history["objective_impatience"].append(objective_impatience)
                if optimizer.track_objective_impatience:
                    postfix["obj_imp"] = objective_impatience
                if optimizer.compute_validation_observed_jd:
                    validation_stats = observed_validation_jeffreys(
                        X_val=X_val,
                        mask_val=mask_val,
                        mu=mu_est,
                        Sigma=Sigma_est,
                        censoring_rule=censoring_rule,
                        rng=rng,
                        eta=optimizer.validation_mask_eta,
                        kernel_epsilon=optimizer.validation_kernel_epsilon,
                        kernel_eig_floor=optimizer.validation_kernel_eig_floor,
                    )
                    validation_value = validation_stats["observed_jeffreys"]
                    history["validation_iteration"].append(int(t))
                    history["validation_observed_jeffreys"].append(validation_value)
                    history["validation_mask_jeffreys"].append(validation_stats["mask_jeffreys"])
                    history["validation_value_jeffreys"].append(validation_stats["value_jeffreys"])
                    history["validation_valid_mask_count"].append(validation_stats["valid_mask_count"])
                    validation_delta = np.nan
                    if optimizer.track_JD_impatience and len(history["validation_observed_jeffreys"]) >= 2:
                        previous_validation = history["validation_observed_jeffreys"][-2]
                        if np.isfinite(validation_value) and np.isfinite(previous_validation):
                            validation_delta = validation_value - previous_validation
                            if validation_delta > 0.0:
                                validation_impatience += 1
                            elif validation_delta < 0.0:
                                validation_impatience = 0
                            if optimizer.validation_impatience_threshold is not None and validation_impatience > optimizer.validation_impatience_threshold:
                                stop_after_iteration = True
                    history["validation_impatience"].append(validation_impatience)
                    postfix["Δval_JD"] = f"{validation_delta:+.6g}"
                    if optimizer.track_JD_impatience:
                        postfix["JD_imp"] = validation_impatience

            if live_plot_handle is not None and (
                t % optimizer.live_plot_every == 0 or t == optimizer.n_iter or stop_after_iteration or validation_recorded
            ):
                live_plot_count = int(compute_error) + int(len(history["objective"]) > 0)
                if live_plot_count > 0:
                    with plt.style.context("dark_background"):
                        fig_live, live_axes = plt.subplots(live_plot_count, 1, figsize=(8, 3.5 * live_plot_count), constrained_layout=True)
                        live_axes = np.atleast_1d(live_axes)
                        fig_live.patch.set_facecolor("black")
                        live_axis_idx = 0
                        if compute_error:
                            live_iterations = np.arange(len(history["mean_squared_l2_error"]))
                            ax_live = live_axes[live_axis_idx]
                            ax_live.plot(live_iterations, history["mean_squared_l2_error"], linewidth=1.5, color="tab:orange")
                            ax_live.set_title("Live Mean Squared L2 Error")
                            ax_live.set_xlabel("Iteration")
                            ax_live.set_ylabel("Mean squared L2 error")
                            ax_live.grid(True, color="0.45", alpha=0.35)
                            live_axis_idx += 1
                        if len(history["objective"]) > 0:
                            objective_iterations_live = np.asarray(history.get("objective_iteration", []), dtype=int)
                            objective_values_live = np.asarray(history["objective"], dtype=float)
                            if objective_iterations_live.size != objective_values_live.size:
                                objective_iterations_live = np.arange(1, objective_values_live.size + 1)
                            ax_live = live_axes[live_axis_idx]
                            ax_live.plot(objective_iterations_live, objective_values_live, linewidth=1.5, color="salmon")
                            ax_live.set_title("Live Validation Objective")
                            ax_live.set_xlabel("Iteration")
                            ax_live.set_ylabel("Objective")
                            ax_live.grid(True, color="0.45", alpha=0.35)
                    live_plot_handle.update(fig_live)
                    plt.close(fig_live)
            if approximation_style == "AUTO":
                postfix["fast_to_slow"] = fast_slow_fallback_count
            if dashboard_iteration_bar is not None:
                dashboard_iteration_bar.set_postfix(postfix, refresh=True)
                dashboard_iteration_bar.update(1)
                if hasattr(progress_dashboard, "heartbeat"):
                    progress_dashboard.heartbeat(f"iteration {t}/{optimizer.n_iter}")
            else:
                iteration_range.set_postfix(postfix, refresh=True)

            if stop_after_iteration:
                break

    finally:
        if live_plot_handle is not None:
            if dashboard_active and hasattr(progress_dashboard, "heartbeat"):
                progress_dashboard.heartbeat("fit complete")
            else:
                live_plot_handle.update(HTML(""))
        if dashboard_iteration_bar is not None:
            close_progress_bar(dashboard_iteration_bar)
        else:
            close_progress_bar(iteration_range)

    if update_style == "SGD":
        final_mu, final_Sigma = canonical_to_raw(bar_nu, bar_T)
        final_nu, final_T = bar_nu, bar_T
    else:
        final_mu, final_Sigma = mu, Sigma
        final_nu, final_T = nu, T

    return FitResult(
        mean=final_mu,
        covariance=final_Sigma,
        history=history,
        nu=final_nu,
        precision=final_T,
        update_style=update_style,
        approximation_style=approximation_style,
    )


In [14]:
# Ablation study helpers


FAMILY_ALIASES = {
    "iso": "iso",
    "isotropic": "iso",
    "isotropic_gaussian": "iso",
    "diagonal": "diag",
    "diag": "diag",
    "diagonal_gaussian": "diag",
    "ar": "ar",
    "ar_correlated": "ar",
    "ar-correlated": "ar",
    "ar_correlated_gaussian": "ar",
    "random": "random",
    "rand": "random",
    "random_gaussian": "random",
    "full": "random",
    "full_covariance": "random",
    "mix": "mix",
    "mixed": "mix",
    "linear_combination": "mix",
    "independent_linear_combination": "mix",
}

FAMILY_LABELS = {
    "iso": "isotropic",
    "diag": "diagonal",
    "ar": "ar_correlated",
    "random": "random",
    "mix": "independent_linear_combination",
}

CENSORING_STYLE_ALIASES = {
    "self": "self",
    "self_censoring": "self",
    "self-censoring": "self",
    "linear": "linear",
    "linear_censoring": "linear",
    "linear-censoring": "linear",
}

INDEPENDENT_VARIABLE_ALIASES = {
    "update_style": "update_style",
    "update style": "update_style",
    "censoring_style": "censoring_style",
    "censoring style": "censoring_style",
    "alpha_min_pair": "alpha_min_pair",
    "alpha_pair_min": "alpha_min_pair",
    "sample_size": "sample_size",
    "sample size": "sample_size",
    "n": "sample_size",
    "batch_size": "batch_size",
    "batch size": "batch_size",
}

ABLATION_DEFAULT_SAMPLE_SIZE = 10**4
ABLATION_DEFAULT_ALPHA_MIN_PAIR = 0.5
ABLATION_DEFAULT_CENSORING_STYLE = "linear"
ABLATION_DEFAULT_UPDATE_STYLE = "SGD"
ABLATION_DEFAULT_APPROXIMATION_STYLE = "AUTO"
ABLATION_DEFAULT_OPTIMIZER = OptimizerConfig(
    n_iter=2**10,
    batch_size=2**5,
    eta_nu0=1e-1,
    eta_T0=1e-2,
    decay_lr=True,
    live_plot=True,
    live_plot_every=5,
    validation_every=2**4,
    validation_impatience_threshold=2**4,
)
ABLATION_DEFAULT_SAMPLER = SamplerConfig(
    K=10**2,
    burn_in=2**10,
    thin=1,
    warm_start=True,
    fast_grid_points=10,
    fast_tilt_samples=2**10,
    fast_min_feasible=10**2,
    fast_max_grid_size=10**4,
)
ABLATION_DEFAULT_PROJECTION = ProjectionConfig(use_A2=False, use_A3=False)
ABLATION_DEFAULT_SHOW_PROGRESS = True


@dataclass(frozen=True)
class AblationProblem:
    problem_class: Tuple[str, int]
    true_mean: np.ndarray
    true_covariance: np.ndarray
    sample_size: int
    X_full: np.ndarray
    censoring_style: str
    alpha_min_pair: float
    censoring_rule: CensoringRule
    true_parameter_metadata: Dict[str, Any]


AblationProblem.alpha_pair_min = property(lambda self: self.alpha_min_pair)


def coerce_rng(rng: Optional[Any] = None) -> np.random.Generator:
    if isinstance(rng, np.random.Generator):
        return rng
    return np.random.default_rng(rng)


def normalize_problem_family(family: str) -> str:
    key = str(family).strip().lower().replace("-", "_").replace(" ", "_")
    if key not in FAMILY_ALIASES:
        raise ValueError(f"Unknown Gaussian family {family!r}. Choose from {sorted(set(FAMILY_ALIASES.values()))}.")
    return FAMILY_ALIASES[key]


def normalize_problem_class(problem_class: Tuple[str, int]) -> Tuple[str, int]:
    family, dimension = problem_class
    family = normalize_problem_family(family)
    dimension = int(dimension)
    if dimension <= 0:
        raise ValueError("Problem dimension must be positive.")
    return family, dimension


def normalize_censoring_style(censoring_style: str) -> str:
    key = str(censoring_style).strip().lower().replace("-", "_").replace(" ", "_")
    if key not in CENSORING_STYLE_ALIASES:
        raise ValueError("censoring_style must be 'self' or 'linear'.")
    return CENSORING_STYLE_ALIASES[key]


def normalize_independent_variable_name(name: str) -> str:
    key = str(name).strip().lower().replace("-", "_")
    key = key.replace(" ", "_") if key not in INDEPENDENT_VARIABLE_ALIASES else key
    key = INDEPENDENT_VARIABLE_ALIASES.get(key, key)
    if key not in set(INDEPENDENT_VARIABLE_ALIASES.values()):
        raise ValueError(
            "independent_variable_name must be one of: update_style, censoring_style, "
            "alpha_min_pair, sample_size, batch_size."
        )
    return key


def validate_alpha_min_pair(alpha_min_pair: float) -> float:
    alpha_min_pair = float(alpha_min_pair)
    if not np.isfinite(alpha_min_pair) or not (0.0 <= alpha_min_pair <= 1.0):
        raise ValueError("alpha_min_pair must be in [0, 1].")
    return alpha_min_pair


def alpha_min_pair_to_visible_probability(alpha_min_pair: float) -> float:
    return 0.5 * (1.0 + validate_alpha_min_pair(alpha_min_pair))


def generate_true_parameters(
    problem_class: Tuple[str, int],
    rng: Optional[Any] = None,
    *,
    mean_scale: float = 1.0,
    variance_low: float = 0.5,
    variance_high: float = 2.0,
    ar_correlation_limit: float = 0.8,
    mix_weight_scale: float = 1.0,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    family, d = normalize_problem_class(problem_class)
    rng = coerce_rng(rng)
    if variance_low <= 0.0 or variance_high <= variance_low:
        raise ValueError("Require 0 < variance_low < variance_high.")
    if not (0.0 < ar_correlation_limit < 1.0):
        raise ValueError("ar_correlation_limit must be in (0, 1).")
    if mix_weight_scale <= 0.0:
        raise ValueError("mix_weight_scale must be positive.")
    if mean_scale < 0.0:
        raise ValueError("mean_scale must be nonnegative.")

    if family == "iso":
        a = float(rng.normal(scale=mean_scale))
        b = float(rng.uniform(variance_low, variance_high))
        mu = np.full(d, a, dtype=float)
        Sigma = b * np.eye(d)
        metadata = {"family": FAMILY_LABELS[family], "a": a, "b": b}
    elif family == "diag":
        mu = rng.normal(scale=mean_scale, size=d)
        variances = rng.uniform(variance_low, variance_high, size=d)
        Sigma = np.diag(variances)
        metadata = {"family": FAMILY_LABELS[family], "mu": mu, "variances": variances}
    elif family == "ar":
        c = float(rng.uniform(-ar_correlation_limit, ar_correlation_limit))
        idx = np.arange(d)
        mu = np.zeros(d, dtype=float)
        Sigma = c ** np.abs(idx[:, None] - idx[None, :])
        metadata = {"family": FAMILY_LABELS[family], "c": c}
    elif family == "random":
        mu = rng.normal(scale=mean_scale, size=d)
        random_matrix = rng.normal(size=(d, d))
        Q, R = np.linalg.qr(random_matrix)
        Q = Q @ np.diag(np.sign(np.diag(R)) + (np.diag(R) == 0.0))
        eigenvalues = rng.uniform(variance_low, variance_high, size=d)
        Sigma = Q @ np.diag(eigenvalues) @ Q.T
        metadata = {"family": FAMILY_LABELS[family], "mu": mu, "eigenvalues": eigenvalues}
    else:
        mu_iso, Sigma_iso, metadata_iso = generate_true_parameters(
            ("iso", d),
            rng,
            mean_scale=mean_scale,
            variance_low=variance_low,
            variance_high=variance_high,
            ar_correlation_limit=ar_correlation_limit,
            mix_weight_scale=mix_weight_scale,
        )
        mu_diag, Sigma_diag, metadata_diag = generate_true_parameters(
            ("diag", d),
            rng,
            mean_scale=mean_scale,
            variance_low=variance_low,
            variance_high=variance_high,
            ar_correlation_limit=ar_correlation_limit,
            mix_weight_scale=mix_weight_scale,
        )
        mu_ar, Sigma_ar, metadata_ar = generate_true_parameters(
            ("ar", d),
            rng,
            mean_scale=mean_scale,
            variance_low=variance_low,
            variance_high=variance_high,
            ar_correlation_limit=ar_correlation_limit,
            mix_weight_scale=mix_weight_scale,
        )
        weights = rng.normal(scale=mix_weight_scale, size=3)
        while np.linalg.norm(weights) <= 1e-12:
            weights = rng.normal(scale=mix_weight_scale, size=3)
        r1, r2, r3 = weights
        mu = r1 * mu_iso + r2 * mu_diag + r3 * mu_ar
        Sigma = r1**2 * Sigma_iso + r2**2 * Sigma_diag + r3**2 * Sigma_ar
        metadata = {
            "family": FAMILY_LABELS[family],
            "weights": weights,
            "components": {"iso": metadata_iso, "diag": metadata_diag, "ar": metadata_ar},
        }

    return np.asarray(mu, dtype=float), eigen_clip_spd(Sigma, lower=1e-10), metadata


def generate_censoring_rule_for_style(
    true_mean: ArrayLike,
    true_covariance: ArrayLike,
    censoring_style: str,
    alpha_min_pair: float,
    rng: Optional[Any] = None,
) -> CensoringRule:
    censoring_style = normalize_censoring_style(censoring_style)
    if censoring_style == "self":
        return generate_self_censoring_rule(true_mean, true_covariance, alpha_min_pair, rng=coerce_rng(rng))
    return generate_linear_censoring_rule(true_mean, true_covariance, alpha_min_pair, rng=coerce_rng(rng))


def modify_censoring_rule_to_alpha_min_pair(
    true_mean: ArrayLike,
    true_covariance: ArrayLike,
    censoring_rule: CensoringRule,
    alpha_min_pair: float,
    tol: float = 1e-12,
) -> CensoringRule:
    mu = np.asarray(true_mean, dtype=float)
    Sigma = symmetrize(true_covariance)
    alpha_min_pair = validate_alpha_min_pair(alpha_min_pair)
    p_visible_target = alpha_min_pair_to_visible_probability(alpha_min_pair)

    if isinstance(censoring_rule, LinearCensoringRule):
        A = censoring_rule.A.copy()
        if p_visible_target >= 1.0:
            target_b = np.full_like(censoring_rule.b, np.inf, dtype=float)
        else:
            tau = norm.ppf(p_visible_target)
            target_b = np.empty_like(censoring_rule.b, dtype=float)
            for j in range(A.shape[0]):
                mean_j = float(A[j] @ mu)
                var_j = float(A[j] @ Sigma @ A[j])
                if var_j <= 0.0 or not np.isfinite(var_j):
                    raise ValueError("Linear-censoring alpha adjustment requires positive finite projected variances.")
                sd_j = np.sqrt(var_j)
                target_b[j] = mean_j + tau * sd_j
        return LinearCensoringRule(A, target_b)

    if not isinstance(censoring_rule, SelfCensoringRule):
        raise TypeError("censoring_rule must be a SelfCensoringRule or LinearCensoringRule.")

    intervals = censoring_rule.intervals.copy()
    target_missing_probability = 1.0 - p_visible_target
    for j in range(len(mu)):
        sd_j = np.sqrt(Sigma[j, j])
        if not np.isfinite(sd_j) or sd_j <= 0.0:
            raise ValueError("Self-censoring alpha adjustment requires positive finite marginal variances.")
        lo, hi = intervals[j]
        lo_cdf = 0.0 if np.isneginf(lo) else float(norm.cdf((lo - mu[j]) / sd_j))
        hi_cdf = 1.0 if np.isposinf(hi) else float(norm.cdf((hi - mu[j]) / sd_j))
        missing_probability = max(0.0, hi_cdf - lo_cdf)
        if abs(missing_probability - target_missing_probability) <= tol:
            continue

        lower_side_is_nearer = abs(lo - mu[j]) <= abs(hi - mu[j])

        def candidate_by_moving_lower() -> Tuple[float, bool]:
            if target_missing_probability <= 0.0:
                return hi, np.isfinite(hi)
            new_lo_cdf = hi_cdf - target_missing_probability
            if new_lo_cdf < -tol or new_lo_cdf > hi_cdf + tol:
                return np.nan, False
            new_lo_cdf = np.clip(new_lo_cdf, np.finfo(float).tiny, 1.0 - np.finfo(float).eps)
            new_lo = mu[j] + sd_j * norm.ppf(new_lo_cdf)
            return new_lo, bool(new_lo <= hi + tol)

        def candidate_by_moving_upper() -> Tuple[float, bool]:
            if target_missing_probability <= 0.0:
                return lo, np.isfinite(lo)
            new_hi_cdf = lo_cdf + target_missing_probability
            if new_hi_cdf < lo_cdf - tol or new_hi_cdf > 1.0 + tol:
                return np.nan, False
            new_hi_cdf = np.clip(new_hi_cdf, np.finfo(float).tiny, 1.0 - np.finfo(float).eps)
            new_hi = mu[j] + sd_j * norm.ppf(new_hi_cdf)
            return new_hi, bool(new_hi >= lo - tol)

        side_order = ("lower", "upper") if lower_side_is_nearer else ("upper", "lower")
        for side in side_order:
            if side == "lower":
                new_lo, feasible = candidate_by_moving_lower()
                if feasible:
                    intervals[j, 0] = new_lo
                    break
            else:
                new_hi, feasible = candidate_by_moving_upper()
                if feasible:
                    intervals[j, 1] = new_hi
                    break
        else:
            raise ValueError("Could not adjust self-censoring interval to the requested alpha_min_pair.")
    return SelfCensoringRule(intervals)


def resize_sample_dataset(
    X_full: Optional[ArrayLike],
    sample_size: int,
    true_mean: ArrayLike,
    true_covariance: ArrayLike,
    rng: Optional[Any] = None,
) -> np.ndarray:
    sample_size = int(sample_size)
    if sample_size <= 0:
        raise ValueError("sample_size must be positive.")
    rng = coerce_rng(rng)
    mu = np.asarray(true_mean, dtype=float)
    Sigma = symmetrize(true_covariance)
    if X_full is None:
        return rng.multivariate_normal(mu, Sigma, size=sample_size)

    X_full = np.asarray(X_full, dtype=float)
    if X_full.ndim != 2 or X_full.shape[1] != len(mu):
        raise ValueError("X_full must have shape (n, d) matching true_mean.")
    if X_full.shape[0] >= sample_size:
        return X_full[:sample_size].copy()
    extra = rng.multivariate_normal(mu, Sigma, size=sample_size - X_full.shape[0])
    return np.vstack([X_full, extra])


def generate_ablation_problem(
    *,
    problem_class: Tuple[str, int],
    sample_size: int,
    censoring_style: str,
    alpha_min_pair: float,
    true_parameters: Optional[Tuple[ArrayLike, ArrayLike]] = None,
    true_mean: Optional[ArrayLike] = None,
    true_covariance: Optional[ArrayLike] = None,
    X_full: Optional[ArrayLike] = None,
    censoring_rule: Optional[CensoringRule] = None,
    rng: Optional[Any] = None,
) -> AblationProblem:
    rng = coerce_rng(rng)
    problem_class = normalize_problem_class(problem_class)
    censoring_style = normalize_censoring_style(censoring_style)
    alpha_min_pair = validate_alpha_min_pair(alpha_min_pair)

    if true_parameters is not None:
        true_mean, true_covariance = true_parameters
    if true_mean is None or true_covariance is None:
        true_mean, true_covariance, true_parameter_metadata = generate_true_parameters(problem_class, rng)
    else:
        true_mean = np.asarray(true_mean, dtype=float)
        true_covariance = eigen_clip_spd(true_covariance, lower=1e-10)
        true_parameter_metadata = {"family": FAMILY_LABELS[problem_class[0]], "source": "provided"}

    X_full = resize_sample_dataset(X_full, sample_size, true_mean, true_covariance, rng)
    if censoring_rule is None or normalize_censoring_style(censoring_rule.kind) != censoring_style:
        censoring_rule = generate_censoring_rule_for_style(true_mean, true_covariance, censoring_style, alpha_min_pair, rng)
    else:
        censoring_rule = modify_censoring_rule_to_alpha_min_pair(true_mean, true_covariance, censoring_rule, alpha_min_pair)

    return AblationProblem(
        problem_class=problem_class,
        true_mean=true_mean,
        true_covariance=true_covariance,
        sample_size=int(sample_size),
        X_full=X_full,
        censoring_style=censoring_style,
        alpha_min_pair=alpha_min_pair,
        censoring_rule=censoring_rule,
        true_parameter_metadata=true_parameter_metadata,
    )


def censoring_rule_to_dict(censoring_rule: CensoringRule) -> Dict[str, Any]:
    if isinstance(censoring_rule, SelfCensoringRule):
        return {"kind": "self", "intervals": censoring_rule.intervals}
    if isinstance(censoring_rule, LinearCensoringRule):
        return {"kind": "linear", "A": censoring_rule.A, "b": censoring_rule.b}
    raise TypeError("Unsupported censoring rule type.")


def json_safe(value: Any) -> Any:
    if is_dataclass(value):
        return json_safe(asdict(value))
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return value if np.isfinite(value) else None
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, Path):
        return str(value)
    return value


def problem_definition_to_dict(problem: AblationProblem) -> Dict[str, Any]:
    return json_safe(
        {
            "problem_class": {
                "family": FAMILY_LABELS[problem.problem_class[0]],
                "dimension": problem.problem_class[1],
            },
            "true_mean": problem.true_mean,
            "true_covariance": problem.true_covariance,
            "sample_size": problem.sample_size,
            "censoring_style": problem.censoring_style,
            "alpha_min_pair": problem.alpha_min_pair,
            "censoring_rule": censoring_rule_to_dict(problem.censoring_rule),
            "true_parameter_metadata": problem.true_parameter_metadata,
        }
    )


def fit_result_to_dict(result: FitResult) -> Dict[str, Any]:
    return json_safe(
        {
            "mean": result.mean,
            "covariance": result.covariance,
            "nu": result.nu,
            "precision": result.precision,
            "history": result.history,
            "update_style": result.update_style,
            "approximation_style": result.approximation_style,
        }
    )


def gaussian_jeffreys_divergence_estimate(
    estimated_mean: ArrayLike,
    estimated_covariance: ArrayLike,
    true_mean: ArrayLike,
    true_covariance: ArrayLike,
) -> float:
    estimated_mean = np.asarray(estimated_mean, dtype=float)
    estimated_covariance = symmetrize(estimated_covariance)
    true_mean = np.asarray(true_mean, dtype=float)
    true_covariance = symmetrize(true_covariance)
    d = len(true_mean)
    mean_delta = estimated_mean - true_mean
    return float(
        0.5
        * (
            np.trace(np.linalg.solve(estimated_covariance, true_covariance))
            + np.trace(np.linalg.solve(true_covariance, estimated_covariance))
            + mean_delta
            @ (
                np.linalg.solve(estimated_covariance, mean_delta)
                + np.linalg.solve(true_covariance, mean_delta)
            )
            - 2.0 * d
        )
    )


def append_jsonl(path: Path, row: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(json_safe(row), separators=(",", ":")) + "\n")
        f.flush()


def load_ablation_results(results_path: str | Path = "ablation_results.jsonl") -> pd.DataFrame:
    rows = []
    path = Path(results_path)
    if not path.exists():
        return pd.DataFrame()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def _problem_for_independent_value(
    base_problem: AblationProblem,
    independent_variable_name: str,
    independent_value: Any,
    rng: np.random.Generator,
) -> AblationProblem:
    if independent_variable_name == "censoring_style":
        return generate_ablation_problem(
            problem_class=base_problem.problem_class,
            true_mean=base_problem.true_mean,
            true_covariance=base_problem.true_covariance,
            sample_size=base_problem.sample_size,
            X_full=base_problem.X_full,
            censoring_style=independent_value,
            alpha_min_pair=base_problem.alpha_min_pair,
            censoring_rule=None,
            rng=rng,
        )
    if independent_variable_name == "alpha_min_pair":
        return generate_ablation_problem(
            problem_class=base_problem.problem_class,
            true_mean=base_problem.true_mean,
            true_covariance=base_problem.true_covariance,
            sample_size=base_problem.sample_size,
            X_full=base_problem.X_full,
            censoring_style=base_problem.censoring_style,
            alpha_min_pair=float(independent_value),
            censoring_rule=base_problem.censoring_rule,
            rng=rng,
        )
    if independent_variable_name == "sample_size":
        return generate_ablation_problem(
            problem_class=base_problem.problem_class,
            true_mean=base_problem.true_mean,
            true_covariance=base_problem.true_covariance,
            sample_size=int(independent_value),
            X_full=base_problem.X_full,
            censoring_style=base_problem.censoring_style,
            alpha_min_pair=base_problem.alpha_min_pair,
            censoring_rule=base_problem.censoring_rule,
            rng=rng,
        )
    return base_problem


def _fit_controls_for_independent_value(
    independent_variable_name: str,
    independent_value: Any,
    *,
    base_update_style: str,
    base_optimizer: OptimizerConfig,
) -> Tuple[str, OptimizerConfig]:
    update_style = base_update_style
    optimizer = base_optimizer
    if independent_variable_name == "update_style":
        update_style = normalize_update_style(independent_value)
    elif independent_variable_name == "batch_size":
        optimizer = replace(base_optimizer, batch_size=int(independent_value))
    return update_style, optimizer


def ablation_validation_split_for_problem(sample_size: int, optimizer: OptimizerConfig) -> Tuple[np.ndarray, Dict[str, Any]]:
    validation_indices = prefix_validation_indices(sample_size, optimizer.validation_fraction)
    split_info = {
        "strategy": "prefix_nested",
        "validation_fraction": float(optimizer.validation_fraction),
        "n_total": int(sample_size),
        "n_validation": int(len(validation_indices)),
        "n_train": int(sample_size - len(validation_indices)),
        "validation_indices_are_prefix": True,
        "validation_prefix_stop": int(len(validation_indices)),
    }
    return validation_indices, split_info


def run_ablation_study(
    *,
    families: Sequence[str],
    dimensions: Sequence[int],
    independent_variable_name: str,
    independent_values: Sequence[Any],
    n_problems_per_class: int,
    results_path: str | Path = "ablation_results.jsonl",
    random_state: Optional[int] = None,
    base_sample_size: int = ABLATION_DEFAULT_SAMPLE_SIZE,
    base_alpha_min_pair: float = ABLATION_DEFAULT_ALPHA_MIN_PAIR,
    base_censoring_style: str = ABLATION_DEFAULT_CENSORING_STYLE,
    base_update_style: str = ABLATION_DEFAULT_UPDATE_STYLE,
    base_approximation_style: str = ABLATION_DEFAULT_APPROXIMATION_STYLE,
    base_optimizer: OptimizerConfig = ABLATION_DEFAULT_OPTIMIZER,
    base_sampler: SamplerConfig = ABLATION_DEFAULT_SAMPLER,
    base_projection: ProjectionConfig = ABLATION_DEFAULT_PROJECTION,
    base_show_progress: bool = ABLATION_DEFAULT_SHOW_PROGRESS,
    catch_errors: bool = True,
    show_ablation_progress: bool = True,
    progress_dashboard: Optional[object] = None,
) -> pd.DataFrame:
    independent_variable_name = normalize_independent_variable_name(independent_variable_name)
    if int(n_problems_per_class) <= 0:
        raise ValueError("n_problems_per_class must be positive.")
    independent_values = list(independent_values)
    if not independent_values:
        raise ValueError("independent_values must contain at least one value.")
    rng = coerce_rng(random_state)
    results_path = Path(results_path)
    rows = []
    problem_classes = [
        normalize_problem_class((family, int(dimension)))
        for family, dimension in itertools.product(families, dimensions)
    ]
    base_problem_sample_size = int(base_sample_size)

    dashboard_active = show_ablation_progress and progress_dashboard is not None
    class_bar = progress_dashboard.bar("classes") if dashboard_active else None
    problem_bar = progress_dashboard.bar("problems") if dashboard_active else None
    value_bar = progress_dashboard.bar("values") if dashboard_active else None

    close_problem_class_iter = False
    if class_bar is not None:
        reset_progress_bar(class_bar, total=len(problem_classes), desc=f"{independent_variable_name} classes")
        problem_class_iter = problem_classes
    elif show_ablation_progress:
        problem_class_iter = tqdm(
            problem_classes,
            desc=f"{independent_variable_name} classes",
            unit="class",
            position=1,
            leave=False,
        )
        close_problem_class_iter = True
    else:
        problem_class_iter = problem_classes

    try:
        for problem_class in problem_class_iter:
            class_postfix = {"family": FAMILY_LABELS[problem_class[0]], "d": problem_class[1]}
            if class_bar is not None:
                class_bar.set_postfix(class_postfix, refresh=True)
                reset_progress_bar(problem_bar, total=int(n_problems_per_class), desc="Problems")
                problem_iter = range(int(n_problems_per_class))
                close_problem_iter = False
            elif show_ablation_progress:
                problem_class_iter.set_postfix(**class_postfix, refresh=False)
                problem_iter = tqdm(
                    range(int(n_problems_per_class)),
                    desc="Problems",
                    unit="problem",
                    position=2,
                    leave=False,
                )
                close_problem_iter = True
            else:
                problem_iter = range(int(n_problems_per_class))
                close_problem_iter = False

            try:
                for problem_index in problem_iter:
                    if problem_bar is not None:
                        problem_bar.set_postfix(problem=problem_index + 1, refresh=True)
                    elif show_ablation_progress:
                        problem_iter.set_postfix(problem=problem_index + 1, refresh=False)

                    base_problem = generate_ablation_problem(
                        problem_class=problem_class,
                        sample_size=base_problem_sample_size,
                        censoring_style=base_censoring_style,
                        alpha_min_pair=base_alpha_min_pair,
                        rng=rng,
                    )
                    sample_size_source_problem = base_problem
                    fit_random_state = int(rng.integers(0, np.iinfo(np.int32).max))

                    close_independent_value_iter = False
                    if value_bar is not None:
                        reset_progress_bar(value_bar, total=len(independent_values), desc="Independent values")
                        independent_value_iter = independent_values
                    elif show_ablation_progress:
                        independent_value_iter = tqdm(
                            independent_values,
                            desc="Independent values",
                            unit="value",
                            position=3,
                            leave=False,
                        )
                        close_independent_value_iter = True
                    else:
                        independent_value_iter = independent_values

                    try:
                        for independent_value in independent_value_iter:
                            if value_bar is not None:
                                value_bar.set_postfix(value=independent_value, refresh=True)
                            elif show_ablation_progress:
                                independent_value_iter.set_postfix(value=independent_value, refresh=False)

                            problem_source = sample_size_source_problem if independent_variable_name == "sample_size" else base_problem
                            run_problem = _problem_for_independent_value(
                                problem_source,
                                independent_variable_name,
                                independent_value,
                                rng,
                            )
                            if independent_variable_name == "sample_size" and run_problem.sample_size > sample_size_source_problem.sample_size:
                                sample_size_source_problem = run_problem
                            update_style, optimizer = _fit_controls_for_independent_value(
                                independent_variable_name,
                                independent_value,
                                base_update_style=base_update_style,
                                base_optimizer=base_optimizer,
                            )
                            validation_indices, validation_split = ablation_validation_split_for_problem(
                                run_problem.sample_size,
                                optimizer,
                            )

                            X_obs, mask = apply_censoring_rule(run_problem.X_full, run_problem.censoring_rule)
                            row = {
                                "status": "started",
                                "problem_index": problem_index,
                                "independent_variable_name": independent_variable_name,
                                "independent_variable_value": independent_value,
                                "problem": problem_definition_to_dict(run_problem),
                                "validation_split": json_safe(validation_split),
                                "fit_random_state": fit_random_state,
                                "fit_controls": json_safe(
                                    {
                                        "update_style": update_style,
                                        "approximation_style": base_approximation_style,
                                        "optimizer": asdict(optimizer),
                                        "sampler": asdict(base_sampler),
                                        "projection": asdict(base_projection),
                                        "show_progress": base_show_progress,
                                        "random_state": fit_random_state,
                                        "validation_split": validation_split,
                                    }
                                ),
                                "observed_fraction": float(mask.mean()),
                                "missing_fraction": float(1.0 - mask.mean()),
                            }
                            started_at = time.perf_counter()
                            try:
                                result = fit_mnar_gaussian(
                                    X_obs,
                                    true_mean=run_problem.true_mean,
                                    true_covariance=run_problem.true_covariance,
                                    update_style=update_style,
                                    approximation_style=base_approximation_style,
                                    censoring_rule=run_problem.censoring_rule,
                                    mask=mask,
                                    optimizer=optimizer,
                                    sampler=base_sampler,
                                    projection=base_projection,
                                    random_state=fit_random_state,
                                    validation_indices=validation_indices,
                                    show_progress=base_show_progress,
                                    progress_position_offset=(4 if show_ablation_progress else 0),
                                    progress_dashboard=progress_dashboard,
                                )
                                elapsed_seconds = time.perf_counter() - started_at
                                jeffreys_divergence = gaussian_jeffreys_divergence_estimate(
                                    result.mean,
                                    result.covariance,
                                    run_problem.true_mean,
                                    run_problem.true_covariance,
                                )
                                row.update(
                                    {
                                        "status": "success",
                                        "elapsed_seconds": elapsed_seconds,
                                        "result": fit_result_to_dict(result),
                                        "summary": json_safe(
                                            {
                                                "jeffreys_divergence": jeffreys_divergence,
                                                "accuracy": 1.0 / (1.0 + jeffreys_divergence),
                                                "final_l2_error": result.history.get("l2_error", [None])[-1],
                                                "final_mean_squared_l2_error": result.history.get("mean_squared_l2_error", [None])[-1],
                                                "final_objective": result.history.get("objective", [None])[-1],
                                                "final_validation_observed_jeffreys": (result.history.get("validation_observed_jeffreys") or [None])[-1],
                                                "final_validation_mask_jeffreys": (result.history.get("validation_mask_jeffreys") or [None])[-1],
                                                "final_validation_value_jeffreys": (result.history.get("validation_value_jeffreys") or [None])[-1],
                                                "n_iterations_recorded": len(result.history.get("theta", [])),
                                            }
                                        ),
                                    }
                                )
                            except Exception as exc:
                                elapsed_seconds = time.perf_counter() - started_at
                                row.update(
                                    {
                                        "status": "error",
                                        "elapsed_seconds": elapsed_seconds,
                                        "error_type": type(exc).__name__,
                                        "error_message": str(exc),
                                    }
                                )
                                append_jsonl(results_path, row)
                                rows.append(json_safe(row))
                                if value_bar is not None:
                                    value_bar.update(1)
                                if not catch_errors:
                                    raise
                                continue

                            append_jsonl(results_path, row)
                            rows.append(json_safe(row))
                            if value_bar is not None:
                                value_bar.update(1)
                    finally:
                        if close_independent_value_iter:
                            close_progress_bar(independent_value_iter)
                        if value_bar is not None:
                            close_progress_bar(value_bar)
                    if problem_bar is not None:
                        problem_bar.update(1)
            finally:
                if close_problem_iter:
                    close_progress_bar(problem_iter)
                if problem_bar is not None:
                    close_progress_bar(problem_bar)
            if class_bar is not None:
                class_bar.update(1)
    finally:
        if close_problem_class_iter:
            close_progress_bar(problem_class_iter)
        if class_bar is not None:
            close_progress_bar(class_bar)
    return pd.DataFrame(rows)


In [15]:
# Comprehensive ablation study run

families_to_test = ["iso", "diag", "ar", "mix", "random"]
dimensions_to_test = [2, 3, 4, 5]
n_problems_per_class = 8
'''
    {
        "independent_variable_name": "update_style",
        "independent_values": ["SGD", "EM"],
        "file_stem": "UpdateStyle_AblationResults",
    },
    {
        "independent_variable_name": "censoring_style",
        "independent_values": ["self", "linear"],
        "file_stem": "CensoringStyle_AblationResults",
    },
    {
        "independent_variable_name": "alpha_pair_min",
        "independent_values": [round(0.5 + 0.05 * k, 2) for k in range(8)],
        "file_stem": "AlphaPairMin_AblationResults",
    },
    {
        "independent_variable_name": "batch_size",
        "independent_values": [2**k for k in range(1, 9)],
        "file_stem": "BatchSize_AblationResults",
    },
    {
        "independent_variable_name": "sample_size",
        "independent_values": np.rint(np.linspace(10**2, 10**5, 8)).astype(int).tolist(),
        "file_stem": "SampleSize_AblationResults",
    },
'''
ablation_study_specs = [
    {
        "independent_variable_name": "update_style",
        "independent_values": ["SGD", "EM"],
        "file_stem": "UpdateStyle_AblationResults",
    },
]

project_dir = Path("/Users/jaceleonswift/Desktop/all PC back-up/NTU/year 4/Final Year Project")
results_dir = project_dir / "ablation_results"
results_dir.mkdir(parents=True, exist_ok=True)

base_random_state = 20260613
all_study_results = {}
study_summaries = []

ablation_dashboard = ProgressDashboard(title="Comprehensive ablation rerun")
ablation_dashboard.display()
study_bar = ablation_dashboard.bar("studies")
reset_progress_bar(study_bar, total=len(ablation_study_specs), desc="Ablation studies")
try:
    for study_index, study_spec in enumerate(ablation_study_specs):
        independent_variable_name = study_spec["independent_variable_name"]
        independent_values = study_spec["independent_values"]
        results_path = results_dir / f"{study_spec['file_stem']}.jsonl"
        expected_rows = len(families_to_test) * len(dimensions_to_test) * n_problems_per_class * len(independent_values)
        study_bar.set_postfix(
            variable=independent_variable_name,
            fits=expected_rows,
            refresh=True,
        )

        study_results = run_ablation_study(
            families=families_to_test,
            dimensions=dimensions_to_test,
            independent_variable_name=independent_variable_name,
            independent_values=independent_values,
            n_problems_per_class=n_problems_per_class,
            results_path=results_path,
            random_state=base_random_state + study_index,
            base_optimizer=ABLATION_DEFAULT_OPTIMIZER,
            base_sampler=ABLATION_DEFAULT_SAMPLER,
            base_projection=ABLATION_DEFAULT_PROJECTION,
            base_show_progress=True,
            catch_errors=True,
            show_ablation_progress=True,
            progress_dashboard=ablation_dashboard,
        )

        all_study_results[independent_variable_name] = study_results
        study_summaries.append(
            {
                "independent_variable_name": normalize_independent_variable_name(independent_variable_name),
                "results_path": str(results_path),
                "expected_rows_this_run": expected_rows,
                "rows_returned_this_run": len(study_results),
                "success_rows_this_run": int((study_results["status"] == "success").sum()) if not study_results.empty else 0,
                "error_rows_this_run": int((study_results["status"] == "error").sum()) if not study_results.empty else 0,
            }
        )
        study_bar.update(1)
finally:
    close_progress_bar(study_bar)
    ablation_dashboard.heartbeat("ablation run complete")

study_summary_df = pd.DataFrame(study_summaries)
display(HTML(f"<b>Results directory:</b> {results_dir}"))
display(study_summary_df)


,independent_variable_name,results_path,expected_rows_this_run,rows_returned_this_run,success_rows_this_run,error_rows_this_run
0,update_style,/Users/jaceleonswift/Desktop/all PC back-up/NT...,320,320,317,3
